# CS-781 Phase 2: V1 Best-Branch Recovery + Validation-Selected Inference

This notebook restores the **V1 full-fine-tuning token-BIO path** used by the earlier `submission.zip` run that scored **0.3223** on the public leaderboard. The recovery-control version later disabled that token-BIO branch and ran the separate character-boundary branch only; the resulting `0.2931` therefore was not a controlled reproduction of the higher-scoring run.

**Plan:** preserve the original V1 checkpoint, tokenizer, document split, token-BIO targets, loss weighting, three-epoch training setup, and checkpoint selection. Add only inference-time comparisons performed on the held-out validation documents: the original window-wise decoder versus averaged logits over overlapping windows, followed by conservative per-label logit-bias calibration. Whichever inference path scores best on validation is used for test predictions. The final cell writes exactly one submission ZIP.

The public leaderboard score of 0.3223 is the best observed reference, not a guarantee that this run will exceed it. The full training and leaderboard evaluation must be executed in Kaggle/Codabench using the real checkpoint and data.

## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [ ]:
import os

HF_TOKEN = os.environ.get("HF_TOKEN")

try:
    from kaggle_secrets import UserSecretsClient
    kaggle_token = UserSecretsClient().get_secret("HF")
    if kaggle_token:
        HF_TOKEN = kaggle_token
except Exception as exc:
    print("Kaggle HF secret was not available in this environment.")

if HF_TOKEN:
    from huggingface_hub import login
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face authentication configured.")
else:
    print("No HF token found. Public dataset access may still work; gated access will require a token and accepted dataset terms.")


## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [ ]:
#VERSION 1: DECODER ONLY MODEL; using GQA+ROPE+RMSNORM

import os
import random
import json
import time
import math
import zipfile
import shutil
import torch
import numpy as np

SEED =42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Basic setup loaded.")
print(f"Random seed: {SEED}")

In [ ]:
#DATASET
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

#for this version i am using just the supreme court training files
TRAIN_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"
TEST_FILE ="Phase_1_Pretraining/Test/test.json"

#for this version i was not sure of the computation timing so taking just 100M tokens
TRAIN_TOKENS = 100_000_000
VAL_TOKENS = 5_000_000

print("Dataset settings loaded.")
print("Training file:", TRAIN_FILE)
print("Training tokens:", f"{TRAIN_TOKENS:,}")
print("Validation tokens:", f"{VAL_TOKENS:,}")

In [ ]:
#context length basically tells us about how many tokens it can look at in one traing example
MODEL_CONTEXT_LENGTH=1024
TRAIN_BLOCK_SIZE = MODEL_CONTEXT_LENGTH
print("CL:", MODEL_CONTEXT_LENGTH)
print("Training block size:", TRAIN_BLOCK_SIZE)

In [ ]:
#ARCHITECTURE
#Using 12 layers for a deeper model
N_LAYER=12
N_EMBD=768 #every token will be represented using 768 numbers inside the model
N_HEAD=12

#Number of key/value heads
#12 query heads will share 4 key heads and 4 value heads
N_KV_HEAD=4

#using normal feed-forward network
FFN_MULTIPLIER=4
DROPOUT=0.0
USE_ROPE= True
USE_RMSNORM = True

#sanity check
print("Model architecture settings loaded.")
print(f"Layers: {N_LAYER}")
print(f"Hidden size: {N_EMBD}")
print(f"Query heads: {N_HEAD}")
print(f"Key/Value heads: {N_KV_HEAD}")

In [ ]:
#TRAINING SETTINGS
PER_DEVICE_BATCH_SIZE=4
#To avoid any error as GPU might not be able to fit the large batch directly
GRADIENT_ACCUMULATION_STEPS=8
LEARNING_RATE = 3e-4
WEIGHT_DECAY=0.1
WARMUP_STEPS=500          #The learning rate inreases slowly instead of immediately using the entire value from start
MAX_GRAD_NORM=1000
EVAL_EVERY_STEPS=1000        #validation loss check frequency
LOG_EVERY_STEPS=100    #Print frequency
SAVE_EVERY_STEPS=1000
MAX_CHECKPOINTS_TO_KEEP=5
USE_AMP =torch.cuda.is_available()
USE_GRADIENT_CHECKPOINTING = False

#sanity check
print("Training settings loaded.")
print("Batch size:", PER_DEVICE_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:",PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Learning rate:", LEARNING_RATE)

In [ ]:
#Generation and output settings

TARGET_GENERATION_WORDS=128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.7            #The starter uses 0.1, but its predictions showed strong repetition, so i am experimenting it
TOP_P =0.95
TOP_K=50 #adding this as an experiment
REPETITION_PENALTY=1.1 #Adding a penalty as the predictions showed heavy repetition

#OUTPUT and CHECKPOINT PATHS

OUTPUT_DIR = "/kaggle/working/cs781_phase1_v1"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
CACHE_DIR = os.path.join(OUTPUT_DIR, "cache")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "final_model")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")
CONFIG_PATH = os.path.join(OUTPUT_DIR, "config.json")

for directory in[
    OUTPUT_DIR,CHECKPOINT_DIR,CACHE_DIR,LOG_DIR,MODEL_OUTPUT_DIR
]:
    os.makedirs(directory, exist_ok=True)

print("Generation and output settings loaded.")
print("Output directory:", OUTPUT_DIR)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Cache directory:", CACHE_DIR)

In [ ]:
# Phase 2 output paths. Keep Phase 2 artifacts separate from Phase 1.
EXPERIMENT_NAME = "V1_FullFT"
PHASE2_ROOT_DIR = "/kaggle/working/cs781_phase2_lner"
PHASE2_OUTPUT_DIR = os.path.join(PHASE2_ROOT_DIR, EXPERIMENT_NAME)
PHASE2_CHECKPOINT_DIR = os.path.join(PHASE2_OUTPUT_DIR, "checkpoints")
PHASE2_PREDICTION_PATH = os.path.join(PHASE2_OUTPUT_DIR, "prediction.json")
PHASE2_SUBMISSION_ZIP = os.path.join(PHASE2_OUTPUT_DIR, "submission.zip")

os.makedirs(PHASE2_CHECKPOINT_DIR, exist_ok=True)

print("Experiment:", EXPERIMENT_NAME)
print("Phase 2 output directory:", PHASE2_OUTPUT_DIR)
print("Phase 2 checkpoint directory:", PHASE2_CHECKPOINT_DIR)

In [ ]:
#Experiment configuration
#this is to keep a track of my experiments and implemntations so that they donot mix up with the future ones

experiment_config = {
    "seed": SEED,

    # Dataset
    "dataset": HF_DATASET_REPO,
    "train_file": TRAIN_FILE,
    "train_tokens": TRAIN_TOKENS,
    "val_tokens": VAL_TOKENS,

    # Sequence length
    "context_length": MODEL_CONTEXT_LENGTH,
    "block_size": TRAIN_BLOCK_SIZE,

    # Model architecture
    "n_layer": N_LAYER,
    "n_embd": N_EMBD,
    "n_head": N_HEAD,
    "n_kv_head": N_KV_HEAD,
    "ffn_multiplier": FFN_MULTIPLIER,
    "dropout": DROPOUT,
    "use_rope": USE_ROPE,
    "use_rmsnorm": USE_RMSNORM,

    # Training
    "batch_size": PER_DEVICE_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": WARMUP_STEPS,
    "max_grad_norm": MAX_GRAD_NORM,
    "eval_every_steps": EVAL_EVERY_STEPS,
    "log_every_steps": LOG_EVERY_STEPS,
    "save_every_steps": SAVE_EVERY_STEPS,
    "max_checkpoints_to_keep": MAX_CHECKPOINTS_TO_KEEP,
    "use_amp": USE_AMP,

    # Generation
    "target_generation_words": TARGET_GENERATION_WORDS,
    "max_generation_tokens": MAX_GENERATION_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "repetition_penalty": REPETITION_PENALTY,
}

## 4. Load Tokenizer

In [ ]:
#tokenizer from A1
import os
import json

BPE_FILENAME = "bpe_en_25k.json"
# Set this explicitly if you have more than one tokenizer dataset attached.
BPE_PATH = ""

if BPE_PATH:
    if not os.path.isfile(BPE_PATH):
        raise FileNotFoundError(f"Configured tokenizer does not exist: {BPE_PATH}")
else:
    bpe_candidates = []
    for root, dirs, files in os.walk("/kaggle/input"):
        if BPE_FILENAME in files:
            bpe_candidates.append(os.path.join(root, BPE_FILENAME))
    bpe_candidates = sorted(set(bpe_candidates))

    if not bpe_candidates:
        raise FileNotFoundError(
            f"{BPE_FILENAME} was not found inside /kaggle/input/. "
            "Attach the exact Phase-1 tokenizer dataset."
        )

    # Multiple copies are okay only if their bytes are identical.
    import hashlib
    def _file_sha256_for_discovery(path):
        h = hashlib.sha256()
        with open(path, "rb") as _handle:
            for _chunk in iter(lambda: _handle.read(1024 * 1024), b""):
                h.update(_chunk)
        return h.hexdigest()
    bpe_hashes = {path: _file_sha256_for_discovery(path) for path in bpe_candidates}
    if len(set(bpe_hashes.values())) > 1:
        raise RuntimeError(
            "Multiple different bpe_en_25k.json files were found. Set BPE_PATH explicitly to the "
            "tokenizer used during Phase 1. Candidates:\n" + "\n".join(bpe_candidates)
        )
    BPE_PATH = bpe_candidates[0]

print("Tokenizer found:", BPE_PATH)


# Load the vocabulary and learned BPE merges.
with open(BPE_PATH, "r", encoding="utf-8") as f:
    bpe_data = json.load(f)

bpe_itos = bpe_data["itos"]
bpe_merges = bpe_data["merges"]
bpe_vocab_size = bpe_data["vocab_size"]

# Make sure we have loaded the tokenizer we actually want.
assert bpe_vocab_size == 25_000
assert len(bpe_itos) == 25_000

print("Vocabulary size:", bpe_vocab_size)
print("Number of merge rules:", len(bpe_merges))


# Token -> ID mapping
bpe_stoi = {
    token: idx
    for idx, token in enumerate(bpe_itos)
}
merge_to_id = {}

for merge_index, merge in enumerate(bpe_merges):
    left_token, right_token = merge

    left_id = bpe_stoi[left_token]
    right_id = bpe_stoi[right_token]

    new_token_id = 261 + merge_index
    merge_to_id[(left_id, right_id)] = new_token_id

PAD_TOKEN_ID = bpe_stoi["<pad>"]
UNK_TOKEN_ID = bpe_stoi["<unk>"]
BOS_TOKEN_ID = bpe_stoi["<s>"]
EOS_TOKEN_ID = bpe_stoi["</s>"]
WORD_BOUNDARY_ID = bpe_stoi["<wb>"]

print("\nSpecial token IDs:")
print("PAD:", PAD_TOKEN_ID)
print("UNK:", UNK_TOKEN_ID)
print("BOS:", BOS_TOKEN_ID)
print("EOS:", EOS_TOKEN_ID)
print("WORD BOUNDARY:", WORD_BOUNDARY_ID)

In [ ]:
#Tokenizer Sanity Check
#Before using the tokenizer on the full legal corpus,let's check a small example and make sure encoding and decoding work correctly.

class ByteLevelBPE25k:
    def __init__(self, itos, stoi, merge_to_id, merges):
        self.itos = itos
        self.stoi = stoi
        self.merge_to_id = merge_to_id
        self.merges = merges
        self.vocab_size = len(itos)

        self.pad_token_id = stoi["<pad>"]
        self.unk_token_id = stoi["<unk>"]
        self.bos_token_id = stoi["<s>"]
        self.eos_token_id = stoi["</s>"]
        self.word_boundary_id = stoi["<wb>"]

    def _encode_word(self, word):
        # Start with the UTF-8 bytes of the word.
        sequence = [
            4 + byte
            for byte in word.encode("utf-8")
        ]

        # Our BPE representation keeps a word-boundary token.
        sequence.append(self.word_boundary_id)

        while len(sequence) > 1:
            best_position = None
            best_token_id = None

            for i in range(len(sequence) - 1):
                pair = (sequence[i], sequence[i + 1])
                new_token_id = self.merge_to_id.get(pair)

                if new_token_id is None:
                    continue

                if (
                    best_token_id is None
                    or new_token_id < best_token_id
                ):
                    best_token_id = new_token_id
                    best_position = i

            if best_position is None:
                break

            i = best_position
            sequence = (
                sequence[:i]
                + [best_token_id]
                + sequence[i + 2:]
            )

        return sequence

    def encode(self, text, add_bos=False, add_eos=False):
        # The original tokenizer was trained word-by-word.
        words = text.split()

        token_ids = []

        if add_bos:
            token_ids.append(self.bos_token_id)

        for word in words:
            token_ids.extend(
                self._encode_word(word)
            )

        if add_eos:
            token_ids.append(self.eos_token_id)

        return token_ids

    def _token_to_bytes(self, token_id):
        token = self.itos[token_id]

        # Raw byte token.
        if token.startswith("<0x") and token.endswith(">"):
            return bytes([int(token[3:-1], 16)])

        if token == "<wb>":
            return b" "

        if token in {"<pad>", "<unk>", "<s>", "</s>"}:
            return b""

        # Learned BPE token.
        if token.startswith("<bpe_"):
            merge_index = token_id - 261

            if 0 <= merge_index < len(self.merges):
                left_token, right_token = self.merges[merge_index]

                left_id = self.stoi[left_token]
                right_id = self.stoi[right_token]

                return (
                    self._token_to_bytes(left_id)
                    + self._token_to_bytes(right_id)
                )

        return b""

    def decode(self, token_ids):
        output = bytearray()

        for token_id in token_ids:
            if not 0 <= int(token_id) < self.vocab_size:
                continue

            output.extend(
                self._token_to_bytes(int(token_id))
            )

        return bytes(output).decode(
            "utf-8",
            errors="replace"
        )


# Creating the tokenizer.
tokenizer = ByteLevelBPE25k(
    itos=bpe_itos,
    stoi=bpe_stoi,
    merge_to_id=merge_to_id,
    merges=bpe_merges
)


# Test it on a small legal-style sentence.
sample_text = (
    "The Supreme Court of India has jurisdiction "
    "over constitutional matters."
)

sample_tokens = tokenizer.encode(sample_text)
decoded_text = tokenizer.decode(sample_tokens)

print("Original text:")
print(sample_text)
print("\nToken IDs:")
print(sample_tokens)
print("\nNumber of tokens:", len(sample_tokens))
print("\nDecoded text:")
print(decoded_text)
print("\nVocabulary size:", tokenizer.vocab_size)
assert tokenizer.vocab_size == 25_000
assert len(sample_tokens) > 0
print("\nTokenizer sanity check passed.")

In [ ]:
# Resolve Phase 2 files from an attached Kaggle dataset first, then from the official Hugging Face repository.
# The Capstone Document maps phase_2 to the Phase_2_LNER folder.
from huggingface_hub import hf_hub_download

PHASE2_HF_PATHS = {
    "fold1-text.json": "Phase_2_LNER/Train/fold1-text.json",
    "fold1-gold.json": "Phase_2_LNER/Train/fold1-gold.json",
    "fold2-text.json": "Phase_2_LNER/Train/fold2-text.json",
    "fold2-gold.json": "Phase_2_LNER/Train/fold2-gold.json",
    "fold3-text.json": "Phase_2_LNER/Test/fold3-text.json",
    "labels.txt": "Phase_2_LNER/Train/labels.txt",
}

PHASE2_FILES = {}
PHASE2_FILE_SOURCES = {}
PHASE2_RESOLUTION_ERRORS = {}

for filename, hub_filename in PHASE2_HF_PATHS.items():
    local_matches = []
    for search_root in ["/kaggle/input", "/kaggle/working"]:
        if os.path.isdir(search_root):
            for root, dirs, files in os.walk(search_root):
                if filename in files:
                    local_matches.append(os.path.join(root, filename))

    local_matches = sorted(set(local_matches))
    if len(local_matches) == 1:
        PHASE2_FILES[filename] = local_matches[0]
        PHASE2_FILE_SOURCES[filename] = "local"
        print(f"LOCAL   {filename}: {local_matches[0]}")
        continue

    if len(local_matches) > 1:
        print(f"MULTIPLE local copies found for {filename}; trying the official Hugging Face path instead.")

    try:
        downloaded_path = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=hub_filename,
            repo_type="dataset",
            token=HF_TOKEN or None,
        )
        PHASE2_FILES[filename] = downloaded_path
        PHASE2_FILE_SOURCES[filename] = "huggingface"
        print(f"HUGGING FACE {filename}: {downloaded_path}")
    except Exception as exc:
        PHASE2_RESOLUTION_ERRORS[filename] = f"{type(exc).__name__}: {exc}"
        print(f"NOT RESOLVED {filename}: {type(exc).__name__}: {exc}")

PHASE2_REQUIRED_FILENAMES = list(PHASE2_HF_PATHS)
PHASE2_MISSING_FILES = [name for name in PHASE2_REQUIRED_FILENAMES if name not in PHASE2_FILES]
PHASE2_DATA_READY = not PHASE2_MISSING_FILES

print("\nPhase 2 data ready:", PHASE2_DATA_READY)
print("Local files:", sum(v == "local" for v in PHASE2_FILE_SOURCES.values()))
print("Hub files:", sum(v == "huggingface" for v in PHASE2_FILE_SOURCES.values()))

if PHASE2_MISSING_FILES:
    missing_details = "\n".join(
        f"- {name} ({PHASE2_HF_PATHS[name]}): {PHASE2_RESOLUTION_ERRORS.get(name, 'not found')}"
        for name in PHASE2_MISSING_FILES
    )
    raise RuntimeError(
        "Could not resolve all Phase 2 files. Verify that the repository is "
        f"accessible, the HF token has accepted any access conditions, and the file paths match the current dataset.\n{missing_details}"
    )


## 5. Download Training Data From Hugging Face

In [ ]:
# from huggingface_hub import hf_hub_download

# train_file_path = hf_hub_download(
#     repo_id=HF_DATASET_REPO,
#     filename=TRAIN_FILE,
#     repo_type="dataset",
#     token=HF_TOKEN
# )

# print("Training file:")
# print(train_file_path)

## 6. Create Train and Validation Tokens Set

In [ ]:
# # 6. Create Train and Validation Token Cache
# import os
# import json
# import random
# import time
# import numpy as np

# CACHE_DIR = os.path.join(
#     OUTPUT_DIR,
#     "token_cache"
# )

# os.makedirs(CACHE_DIR, exist_ok=True)

# TRAIN_TOKEN_PATH = os.path.join(
#     CACHE_DIR,
#     "train_tokens.bin"
# )

# VAL_TOKEN_PATH = os.path.join(
#     CACHE_DIR,
#     "val_tokens.bin"
# )

# TOKEN_CACHE_META = os.path.join(
#     CACHE_DIR,
#     "cache_metadata.json"
# )

# # Load the official training corpus
# with open(train_file_path, "r", encoding="utf-8") as f:
#     records = json.load(f)

# print("Number of records:", len(records))

# # Resume from an existing cache if possible

# train_count = 0
# val_count = 0
# start_record = 0

# if (
#     os.path.exists(TRAIN_TOKEN_PATH)
#     and os.path.exists(VAL_TOKEN_PATH)
#     and os.path.exists(TOKEN_CACHE_META)
# ):
#     with open(TOKEN_CACHE_META, "r", encoding="utf-8") as f:
#         cache_meta = json.load(f)

#     # Make sure this cache belongs to the same experiment.
#     if (
#         cache_meta.get("train_tokens") == TRAIN_TOKENS
#         and cache_meta.get("val_tokens") == VAL_TOKENS
#         and cache_meta.get("vocab_size") == tokenizer.vocab_size
#         and cache_meta.get("seed") == SEED
#     ):
#         start_record = cache_meta["next_record"]
#         train_count = cache_meta["train_count"]
#         val_count = cache_meta["val_count"]

#         print("Existing token cache found.")
#         print("Resuming from record:", start_record)
#         print("Train tokens already cached:", f"{train_count:,}")
#         print("Validation tokens already cached:", f"{val_count:,}")

#     else:
#         print("Existing cache does not match the current settings.")
#         print("Starting a fresh token cache.")

#         os.remove(TRAIN_TOKEN_PATH)
#         os.remove(VAL_TOKEN_PATH)
#         os.remove(TOKEN_CACHE_META)

# else:
#     print("No existing token cache found.")
#     print("Starting tokenization from the beginning.")

# # Open cache files in append mode

# train_file = open(
#     TRAIN_TOKEN_PATH,
#     "ab"
# )

# val_file = open(
#     VAL_TOKEN_PATH,
#     "ab"
# )

# # Tokenize the corpus
# # We keep the same record-level random split used earlier:roughly 95% for training and 5% for validation.The seed and record index are saved, so the split stays fixed when the process is resumed.

# random.seed(SEED)

# # Recreate the same random decisions for records that were
# # already processed before an interruption.
# for _ in range(start_record):
#     random.random()

# start_time = time.time()

# for record_index in range(start_record, len(records)):

#     # Stop once both requested token budgets are reached.
#     if (
#         train_count >= TRAIN_TOKENS
#         and val_count >= VAL_TOKENS
#     ):
#         break

#     text = records[record_index]["text"]

#     # Use our custom BPE-25k tokenizer.
#     tokens = tokenizer.encode(
#         text,
#         add_bos=False,
#         add_eos=False
#     )

#     # Decide whether this record belongs to validation.
#     is_validation = random.random() < 0.05

#     if is_validation and val_count < VAL_TOKENS:

#         remaining = VAL_TOKENS - val_count
#         tokens = tokens[:remaining]

#         if tokens:
#             token_array = np.asarray(
#                 tokens,
#                 dtype=np.uint32
#             )

#             val_file.write(
#                 token_array.tobytes()
#             )

#             val_count += len(token_array)

#     elif not is_validation and train_count < TRAIN_TOKENS:

#         remaining = TRAIN_TOKENS - train_count
#         tokens = tokens[:remaining]

#         if tokens:
#             token_array = np.asarray(
#                 tokens,
#                 dtype=np.uint32
#             )

#             train_file.write(
#                 token_array.tobytes()
#             )

#             train_count += len(token_array)


#     # Save progress periodically so an interruption does notforce us to start tokenization from zero.
#     if (
#         (record_index + 1) % 100 == 0
#         or train_count >= TRAIN_TOKENS
#         or val_count >= VAL_TOKENS
#     ):

#         train_file.flush()
#         val_file.flush()

#         cache_meta = {
#             "train_tokens": TRAIN_TOKENS,
#             "val_tokens": VAL_TOKENS,
#             "train_count": train_count,
#             "val_count": val_count,
#             "next_record": record_index + 1,
#             "vocab_size": tokenizer.vocab_size,
#             "seed": SEED
#         }

#         with open(
#             TOKEN_CACHE_META,
#             "w",
#             encoding="utf-8"
#         ) as f:
#             json.dump(
#                 cache_meta,
#                 f,
#                 indent=2
#             )

#     # Print progress every 100 records.
#     if (record_index + 1) % 100 == 0:

#         elapsed = time.time() - start_time

#         print(
#             f"Record {record_index + 1:,}/{len(records):,} | "
#             f"Train: {train_count:,}/{TRAIN_TOKENS:,} | "
#             f"Val: {val_count:,}/{VAL_TOKENS:,} | "
#             f"Time: {elapsed / 60:.1f} min"
#         )


# train_file.flush()
# val_file.flush()

# train_file.close()
# val_file.close()

# # Final checks

# assert train_count == TRAIN_TOKENS, (
#     f"Expected {TRAIN_TOKENS:,} train tokens, "
#     f"got {train_count:,}"
# )

# assert val_count == VAL_TOKENS, (
#     f"Expected {VAL_TOKENS:,} validation tokens, "
#     f"got {val_count:,}"
# )

# print("\nTokenization complete.")
# print("Train tokens:", f"{train_count:,}")
# print("Validation tokens:", f"{val_count:,}")
# print("Train cache:", TRAIN_TOKEN_PATH)
# print("Validation cache:", VAL_TOKEN_PATH)

## 7. Create Token Blocks

In [ ]:
# # Create Dataset From Cached Tokens

# import numpy as np
# import torch
# from torch.utils.data import Dataset

# class LegalTokenDataset(Dataset):
#     def __init__(
#         self,
#         token_path,
#         num_tokens,
#         block_size,
#         stride=1
#     ):
#         self.token_path = token_path
#         self.num_tokens = num_tokens
#         self.block_size = block_size
#         self.stride = stride

#         # The file stays on disk and is accessed only when
#         # a particular training example is requested.
#         self.tokens = np.memmap(
#             token_path,
#             dtype=np.uint32,
#             mode="r",
#             shape=(num_tokens,)
#         )

#     def __len__(self):
#         return 1 + (
#             self.num_tokens
#             - self.block_size
#             - 1
#         ) // self.stride

#     def __getitem__(self, index):
#         start = index * self.stride
#         end = start + self.block_size + 1

#         tokens = self.tokens[start:end].astype(
#             np.int64
#         )

#         input_ids = torch.from_numpy(
#             tokens[:-1]
#         )

#         labels = torch.from_numpy(
#             tokens[1:]
#         )

#         return {
#             "input_ids": input_ids,
#             "labels": labels
#         }

# train_dataset = LegalTokenDataset(
#     token_path=TRAIN_TOKEN_PATH,
#     num_tokens=TRAIN_TOKENS,
#     block_size=TRAIN_BLOCK_SIZE,
#     stride=1
# )

# val_dataset = LegalTokenDataset(
#     token_path=VAL_TOKEN_PATH,
#     num_tokens=VAL_TOKENS,
#     block_size=TRAIN_BLOCK_SIZE,
#     stride=TRAIN_BLOCK_SIZE
# )

# print("Train examples:", f"{len(train_dataset):,}")
# print("Validation examples:", f"{len(val_dataset):,}")
# print("Sequence length:", TRAIN_BLOCK_SIZE)

## 8. Create Datasets

In [ ]:
# from torch.utils.data import DataLoader

# train_loader = DataLoader(
#     train_dataset,
#     batch_size=PER_DEVICE_BATCH_SIZE,
#     shuffle=True,
#     num_workers=2,
#     pin_memory=True,
#     drop_last=True
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=PER_DEVICE_BATCH_SIZE,
#     shuffle=False,
#     num_workers=2,
#     pin_memory=True,
#     drop_last=True
# )

# print("Train batches per epoch:", len(train_loader))
# print("Validation batches:", len(val_loader))
# print("Batch size:", PER_DEVICE_BATCH_SIZE)
# print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)

In [ ]:
# #SANITY CHECK
# batch = next(iter(train_loader))

# print("Input shape:", batch["input_ids"].shape)
# print("Label shape:", batch["labels"].shape)

# print(
#     "Minimum token ID:",
#     batch["input_ids"].min().item()
# )

# print(
#     "Maximum token ID:",
#     batch["input_ids"].max().item()
# )

# assert batch["input_ids"].shape == (
#     PER_DEVICE_BATCH_SIZE,
#     TRAIN_BLOCK_SIZE
# )
# assert batch["labels"].shape == (
#     PER_DEVICE_BATCH_SIZE,
#     TRAIN_BLOCK_SIZE
# )
# assert batch["input_ids"].min() >= 0
# assert batch["input_ids"].max() < tokenizer.vocab_size

# print("\nDataLoader sanity check passed.")

## 9. Define The Model

In [ ]:
# 9. V1 Decoder-Only Transformer
# This is our first architecture built from scratch.
# V1 uses:
# - RMSNorm
# - RoPE
# - Grouped Query Attention
# - Causal self-attention
# - Standard GELU FFN
# - Residual connections
# - KV cache for autoregressive generation

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# RMSNorm
# ---------------------------------------------------------

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(
            dim=-1,
            keepdim=True
        )

        x = x * torch.rsqrt(
            rms + self.eps
        )

        return self.weight * x


# ---------------------------------------------------------
# Rotary Positional Embeddings
# ---------------------------------------------------------

class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=10000
    ):
        super().__init__()

        self.head_dim = head_dim
        self.max_seq_len = max_seq_len

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2
                ).float()
                / head_dim
            )
        )

        self.register_buffer(
            "inv_freq",
            inv_freq,
            persistent=False
        )

        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        positions = torch.arange(
            seq_len,
            device=self.inv_freq.device
        ).float()

        freqs = torch.outer(
            positions,
            self.inv_freq
        )

        self.register_buffer(
            "cos_cached",
            freqs.cos(),
            persistent=False
        )

        self.register_buffer(
            "sin_cached",
            freqs.sin(),
            persistent=False
        )

    def forward(
        self,
        q,
        k,
        position_offset=0
    ):
        seq_len = q.size(-2)

        required_len = (
            position_offset + seq_len
        )

        if required_len > self.max_seq_len:
            self._build_cache(required_len)
            self.max_seq_len = required_len

        cos = self.cos_cached[
            position_offset:
            position_offset + seq_len
        ]

        sin = self.sin_cached[
            position_offset:
            position_offset + seq_len
        ]

        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)

        q = self._apply_rotary(
            q,
            cos,
            sin
        )

        k = self._apply_rotary(
            k,
            cos,
            sin
        )

        return q, k

    @staticmethod
    def _apply_rotary(x, cos, sin):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]

        rotated = torch.stack(
            (-x2, x1),
            dim=-1
        ).flatten(-2)

        cos_full = torch.repeat_interleave(
            cos,
            2,
            dim=-1
        )

        sin_full = torch.repeat_interleave(
            sin,
            2,
            dim=-1
        )

        return (
            x * cos_full
            + rotated * sin_full
        )


# ---------------------------------------------------------
# Grouped Query Attention
# ---------------------------------------------------------

class GQAAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0
        assert n_head % n_kv_head == 0

        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head

        self.kv_repeat = (
            n_head // n_kv_head
        )

        self.q_proj = nn.Linear(
            n_embd,
            n_head * self.head_dim,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_kv_head * self.head_dim,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.rope = RotaryEmbedding(
            head_dim=self.head_dim,
            max_seq_len=max_seq_len
        )

        self.dropout = dropout

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B,
            T,
            self.n_head,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_head,
            self.head_dim
        ).transpose(1, 2)

        past_len = 0

        if past_key_value is not None:
            past_k, past_v = past_key_value
            past_len = past_k.size(-2)

        # Apply RoPE to the new Q and K positions.
        q, k = self.rope(
            q,
            k,
            position_offset=past_len
        )

        # Add the new K/V to the cached K/V during generation.
        if past_key_value is not None:
            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        # GQA: each KV head is shared by a group of query heads.
        if self.kv_repeat > 1:
            k = k.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

            v = v.repeat_interleave(
                self.kv_repeat,
                dim=1
            )

        # During training, each token can only attend to
        # itself and tokens before it.
        #
        # IMPORTANT:
        # For scaled_dot_product_attention, True means
        # "this position is allowed to attend".
        if past_key_value is None:
            attn_mask = torch.tril(
                torch.ones(
                    T,
                    T,
                    device=x.device,
                    dtype=torch.bool
                )
            )
        else:
            # During cached decoding, all cached positions
            # are valid keys for the current token.
            attn_mask = None

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = y.transpose(
            1,
            2
        ).contiguous()

        y = y.view(
            B,
            T,
            C
        )

        y = self.out_proj(y)

        return y, present


# ---------------------------------------------------------
# Standard FFN
# ---------------------------------------------------------

class FeedForward(nn.Module):
    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        hidden_dim = int(
            n_embd * multiplier
        )

        self.fc1 = nn.Linear(
            n_embd,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_embd
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)

        return x


# ---------------------------------------------------------
# Transformer Block
# ---------------------------------------------------------

class TransformerBlock(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = RMSNorm(n_embd)

        self.attention = GQAAttention(
            n_embd=n_embd,
            n_head=n_head,
            n_kv_head=n_kv_head,
            max_seq_len=max_seq_len,
            dropout=dropout
        )

        self.norm2 = RMSNorm(n_embd)

        self.ffn = FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False
    ):
        # Pre-norm attention followed by a residual connection.
        attn_output, present = self.attention(
            self.norm1(x),
            past_key_value=past_key_value,
            use_cache=use_cache
        )

        x = x + attn_output

        # Pre-norm FFN followed by a residual connection.
        x = x + self.ffn(
            self.norm2(x)
        )

        return x, present


# ---------------------------------------------------------
# V1 Language Model
# ---------------------------------------------------------

class V1LegalLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd,
        n_layer,
        n_head,
        n_kv_head,
        max_seq_len,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                n_embd=n_embd,
                n_head=n_head,
                n_kv_head=n_kv_head,
                max_seq_len=max_seq_len,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_layer)
        ])

        self.final_norm = RMSNorm(
            n_embd
        )

        # Tie the LM head to the token embedding weights.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def forward(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False
    ):
        x = self.token_embedding(
            input_ids
        )

        presents = [] if use_cache else None

        for layer_index, block in enumerate(
            self.blocks
        ):
            past = None

            if past_key_values is not None:
                past = past_key_values[
                    layer_index
                ]

            x, present = block(
                x,
                past_key_value=past,
                use_cache=use_cache
            )

            if use_cache:
                presents.append(
                    present
                )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        return logits, presents

In [ ]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
model = V1LegalLM(
    vocab_size=tokenizer.vocab_size,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_kv_head=N_KV_HEAD,
    max_seq_len=MODEL_CONTEXT_LENGTH,
    ffn_multiplier=FFN_MULTIPLIER,
    dropout=DROPOUT
).to(DEVICE)

print("V1 model created successfully.")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Layers: {N_LAYER}")
print(f"Embedding dimension: {N_EMBD}")
print(f"Attention heads: {N_HEAD}")
print(f"KV heads: {N_KV_HEAD}")
print(f"Context length: {MODEL_CONTEXT_LENGTH}")

In [ ]:
# Restore the trained Phase 1 V1 weights before any Phase 2 adaptation.
# Set V1_CHECKPOINT_PATH to an exact path if automatic discovery is ambiguous.
V1_CHECKPOINT_PATH = ""

if V1_CHECKPOINT_PATH:
    if not os.path.isfile(V1_CHECKPOINT_PATH):
        raise FileNotFoundError(
            f"The configured V1 checkpoint does not exist: {V1_CHECKPOINT_PATH}"
        )
    checkpoint_path = V1_CHECKPOINT_PATH
else:
    checkpoint_search_roots = ["/kaggle/input", "/kaggle/working"]
    checkpoint_candidates = []
    target_names = ["v1_model.pt"]  # Never guess from generic best.pt/latest.pt filenames.

    for search_root in checkpoint_search_roots:
        if os.path.isdir(search_root):
            for root, dirs, files in os.walk(search_root):
                for target_name in target_names:
                    if target_name in files:
                        checkpoint_candidates.append(
                            (target_names.index(target_name),
                             os.path.join(root, target_name))
                        )

    checkpoint_candidates.sort(key=lambda item: (item[0], item[1]))
    if not checkpoint_candidates:
        raise FileNotFoundError(
            "Could not find the Phase-1 V1 checkpoint v1_model.pt under "
            "/kaggle/input or /kaggle/working. Attach your trained Phase 1 V1 "
            "checkpoint as a Kaggle input, or set V1_CHECKPOINT_PATH explicitly."
        )

    best_priority = checkpoint_candidates[0][0]
    best_matches = [
        path for priority, path in checkpoint_candidates
        if priority == best_priority
    ]
    if len(best_matches) > 1:
        raise RuntimeError(
            "Multiple checkpoint files with the same preferred filename were found. "
            "Set V1_CHECKPOINT_PATH to the exact checkpoint you intend to use:\n"
            + "\n".join(best_matches)
        )
    checkpoint_path = best_matches[0]

print("Loading V1 checkpoint:", checkpoint_path)
checkpoint = torch.load(checkpoint_path, map_location="cpu")

if not isinstance(checkpoint, dict):
    raise TypeError("Expected the V1 checkpoint to be a dictionary.")

checkpoint_config = checkpoint.get("config")
expected_config = {
    "vocab_size": tokenizer.vocab_size,
    "n_embd": N_EMBD,
    "n_layer": N_LAYER,
    "n_head": N_HEAD,
    "n_kv_head": N_KV_HEAD,
    "ffn_multiplier": FFN_MULTIPLIER,
    "dropout": DROPOUT,
    "context_length": MODEL_CONTEXT_LENGTH,
}

if checkpoint_config is not None:
    mismatches = {}
    for key, expected_value in expected_config.items():
        if key in checkpoint_config and checkpoint_config[key] != expected_value:
            mismatches[key] = {
                "checkpoint": checkpoint_config[key],
                "notebook": expected_value,
            }
    if mismatches:
        raise ValueError(
            "Checkpoint configuration does not match this notebook: "
            + json.dumps(mismatches, indent=2)
        )
    print("Checkpoint configuration checks passed.")
else:
    print("No saved config found in checkpoint; strict weight-shape checks will be used.")

if "model_state_dict" in checkpoint:
    v1_state_dict = checkpoint["model_state_dict"]
elif all(
    isinstance(key, str) and torch.is_tensor(value)
    for key, value in checkpoint.items()
):
    # Support a raw state_dict checkpoint.
    v1_state_dict = checkpoint
else:
    raise KeyError(
        "Checkpoint has no 'model_state_dict' and is not a raw model state_dict. "
        f"Available top-level keys: {list(checkpoint.keys())}"
    )

# strict=True deliberately prevents silent partial or mismatched loading.
model.load_state_dict(v1_state_dict, strict=True)
model.to(DEVICE)
model.eval()

del v1_state_dict
del checkpoint
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Trained Phase 1 V1 weights restored successfully.")
print("Model device:", next(model.parameters()).device)
print("Model in evaluation mode:", not model.training)

## Phase 2: Load and validate the official L-NER annotations

This section loads the two labelled folds and fold-3 test documents, checks document IDs and annotation formats, and creates a reproducible held-out validation split. It does not define or train an NER model.

In [ ]:
from collections import Counter

EXPECTED_PHASE2_LABELS = {
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
}


def _load_json_file(filename):
    with open(PHASE2_FILES[filename], "r", encoding="utf-8") as handle:
        return json.load(handle)


def _normalise_text_map(raw_data, filename):
    # The Phase 2 specification describes text files as document-ID -> full text mappings.
    if not isinstance(raw_data, dict):
        raise TypeError(f"{filename} should contain a JSON object keyed by document ID; got {type(raw_data).__name__}.")
    result = {}
    for document_id, value in raw_data.items():
        if isinstance(value, str):
            text_value = value
        elif isinstance(value, dict) and isinstance(value.get("text"), str):
            # Supports a wrapped record if the published file uses that variant.
            text_value = value["text"]
        else:
            raise TypeError(
                f"Unexpected text record for document {document_id!r} in {filename}: {type(value).__name__}."
            )
        result[str(document_id)] = text_value
    return result


def _first_present(record, keys, field_name, document_id):
    for key in keys:
        if key in record:
            return record[key]
    raise KeyError(
        f"Missing {field_name!r} field for document {document_id!r}. Expected one of {keys}; got keys {list(record)}."
    )


def _normalise_entities_for_document(raw_entities, document_id, filename):
    # Supports the published [start, end, label, optional_text] format and dictionary variants.
    if raw_entities is None:
        return []
    if isinstance(raw_entities, dict):
        if "entities" in raw_entities:
            raw_entities = raw_entities["entities"]
        elif any(k in raw_entities for k in ("start", "start_char", "start_offset")):
            raw_entities = [raw_entities]
        else:
            raise TypeError(
                f"Unexpected annotation object for document {document_id!r} in {filename}: {list(raw_entities)[:12]}"
            )
    if not isinstance(raw_entities, (list, tuple)):
        raise TypeError(
            f"Annotations for document {document_id!r} in {filename} should be a list; got {type(raw_entities).__name__}."
        )

    entities = []
    for entity_index, entity in enumerate(raw_entities):
        if isinstance(entity, dict):
            start = _first_present(entity, ("start", "start_char", "start_offset", "begin"), "start", document_id)
            end = _first_present(entity, ("end", "end_char", "end_offset", "stop"), "end", document_id)
            label = _first_present(entity, ("label", "entity_label", "type", "entity_type"), "label", document_id)
            entity_text = entity.get("text", entity.get("entity_text", entity.get("surface")))
        elif isinstance(entity, (list, tuple)) and len(entity) >= 3:
            start, end, label = entity[:3]
            entity_text = entity[3] if len(entity) >= 4 else None
        else:
            raise TypeError(
                f"Unexpected entity at index {entity_index} for document {document_id!r} in {filename}: {entity!r}"
            )

        try:
            start = int(start)
            end = int(end)
        except (TypeError, ValueError) as exc:
            raise TypeError(
                f"Non-integer span boundaries for document {document_id!r}, entity {entity_index}: {entity!r}"
            ) from exc
        if not isinstance(label, str):
            raise TypeError(f"Entity label must be a string, got {type(label).__name__}: {entity!r}")
        entities.append({"start": start, "end": end, "label": label, "text": entity_text})
    return entities


def _normalise_gold_map(raw_data, filename):
    if not isinstance(raw_data, dict):
        raise TypeError(f"{filename} should contain a JSON object keyed by document ID; got {type(raw_data).__name__}.")
    return {
        str(document_id): _normalise_entities_for_document(entities, str(document_id), filename)
        for document_id, entities in raw_data.items()
    }


with open(PHASE2_FILES["labels.txt"], "r", encoding="utf-8") as handle:
    labels_lines = [line.strip() for line in handle if line.strip() and not line.strip().startswith("#")]

# Accept either one label per line or a label followed by a human-readable description.
LABELS = []
for line in labels_lines:
    first_token = line.split()[0]
    if first_token in EXPECTED_PHASE2_LABELS:
        candidate = first_token
    elif line in EXPECTED_PHASE2_LABELS:
        candidate = line
    else:
        continue
    if candidate not in LABELS:
        LABELS.append(candidate)

if set(LABELS) != EXPECTED_PHASE2_LABELS:
    raise ValueError(
        "labels.txt did not resolve to the expected 12 Phase 2 labels. "
        f"Found={LABELS}; missing={sorted(EXPECTED_PHASE2_LABELS - set(LABELS))}; "
        f"unexpected={sorted(set(LABELS) - EXPECTED_PHASE2_LABELS)}"
    )

FOLD1_TEXTS = _normalise_text_map(_load_json_file("fold1-text.json"), "fold1-text.json")
FOLD1_GOLD = _normalise_gold_map(_load_json_file("fold1-gold.json"), "fold1-gold.json")
FOLD2_TEXTS = _normalise_text_map(_load_json_file("fold2-text.json"), "fold2-text.json")
FOLD2_GOLD = _normalise_gold_map(_load_json_file("fold2-gold.json"), "fold2-gold.json")
TEST_TEXTS = _normalise_text_map(_load_json_file("fold3-text.json"), "fold3-text.json")

for fold_name, texts, gold in (
    ("fold1", FOLD1_TEXTS, FOLD1_GOLD),
    ("fold2", FOLD2_TEXTS, FOLD2_GOLD),
):
    missing_gold = sorted(set(texts) - set(gold))
    orphan_gold = sorted(set(gold) - set(texts))
    if missing_gold or orphan_gold:
        raise ValueError(
            f"{fold_name} text/gold document IDs do not match. "
            f"Missing gold for {len(missing_gold)} IDs; gold without text for {len(orphan_gold)} IDs. "
            f"First missing={missing_gold[:5]}, first orphan={orphan_gold[:5]}"
        )

TEXTS_BY_ID = {}
ANNOTATIONS_BY_ID = {}
for fold_name, texts, gold in (
    ("fold1", FOLD1_TEXTS, FOLD1_GOLD),
    ("fold2", FOLD2_TEXTS, FOLD2_GOLD),
):
    overlap = set(TEXTS_BY_ID) & set(texts)
    if overlap:
        raise ValueError(
            f"Document IDs occur in both training folds ({len(overlap)} overlaps). "
            f"Example IDs: {sorted(overlap)[:10]}"
        )
    TEXTS_BY_ID.update(texts)
    ANNOTATIONS_BY_ID.update(gold)

TEST_OVERLAP = set(TEXTS_BY_ID) & set(TEST_TEXTS)
if TEST_OVERLAP:
    raise ValueError(
        f"Training and test document IDs overlap ({len(TEST_OVERLAP)} IDs). Example: {sorted(TEST_OVERLAP)[:10]}"
    )

# Strictly validate offsets and labels before tokenization; do not silently repair annotations.
ANNOTATION_COUNTS = Counter()
INVALID_SPANS = []
ENTITY_TEXT_MISMATCHES = []
for document_id, entities in ANNOTATIONS_BY_ID.items():
    source_text = TEXTS_BY_ID[document_id]
    for entity_index, entity in enumerate(entities):
        start, end, label = entity["start"], entity["end"], entity["label"]
        ANNOTATION_COUNTS[label] += 1
        if label not in EXPECTED_PHASE2_LABELS:
            INVALID_SPANS.append((document_id, entity_index, "unknown_label", label))
        if not (0 <= start < end <= len(source_text)):
            INVALID_SPANS.append((document_id, entity_index, "invalid_offsets", start, end, len(source_text)))
        elif entity["text"] is not None and source_text[start:end] != entity["text"]:
            ENTITY_TEXT_MISMATCHES.append((document_id, entity_index, entity["text"], source_text[start:end]))

if INVALID_SPANS:
    raise ValueError(
        f"Found {len(INVALID_SPANS)} invalid annotations. Examples: {INVALID_SPANS[:10]}"
    )

print("Official labels:", LABELS)
print(f"Fold 1: {len(FOLD1_TEXTS):,} documents, {sum(map(len, FOLD1_GOLD.values())):,} entities")
print(f"Fold 2: {len(FOLD2_TEXTS):,} documents, {sum(map(len, FOLD2_GOLD.values())):,} entities")
print(f"Combined training pool: {len(TEXTS_BY_ID):,} documents")
print(f"Fold 3 test documents: {len(TEST_TEXTS):,}")
print("Entity counts:")
for label in LABELS:
    print(f"  {label:10s} {ANNOTATION_COUNTS[label]:,}")
print("Optional entity_text mismatches (reported only, offsets preserved):", len(ENTITY_TEXT_MISMATCHES))
if ENTITY_TEXT_MISMATCHES:
    print("Mismatch examples:", ENTITY_TEXT_MISMATCHES[:3])

# Deterministic document-level holdout; keep each document entirely in one split.
all_document_ids = list(TEXTS_BY_ID)
random.Random(SEED).shuffle(all_document_ids)
if len(all_document_ids) < 2:
    raise ValueError("At least two labelled documents are required to create train/validation splits.")
VALIDATION_FRACTION = 0.10
num_validation = max(1, int(round(len(all_document_ids) * VALIDATION_FRACTION)))
num_validation = min(num_validation, len(all_document_ids) - 1)
VAL_DOC_IDS = all_document_ids[:num_validation]
TRAIN_DOC_IDS = all_document_ids[num_validation:]

print(f"\nDocument split: train={len(TRAIN_DOC_IDS):,}, validation={len(VAL_DOC_IDS):,}")
print("Train/validation ID overlap:", len(set(TRAIN_DOC_IDS) & set(VAL_DOC_IDS)))
assert not (set(TRAIN_DOC_IDS) & set(VAL_DOC_IDS))


## Verify tokenizer-compatible character offsets

The V1 tokenizer splits on whitespace, encodes UTF-8 bytes, and applies the stored merge rules. The helper below reproduces those same ID decisions while tracking each token's source-character span. It checks that its token IDs exactly match the existing tokenizer before any alignment result is trusted.

In [ ]:
import re


def _merge_two_offset_items(left_item, right_item, merged_token_id):
    spans = [
        (left_item[1], left_item[2]),
        (right_item[1], right_item[2]),
    ]
    covered = [(start, end) for start, end in spans if start is not None and end is not None]
    if not covered:
        return (merged_token_id, None, None)
    return (
        merged_token_id,
        min(start for start, _ in covered),
        max(end for _, end in covered),
    )


def _encode_word_with_offsets(word, word_start, tokenizer):
    items = []
    char_position = word_start
    for character in word:
        char_start = char_position
        char_end = char_position + len(character)
        for byte_value in character.encode("utf-8"):
            # This mirrors the existing ByteLevelBPE25k._encode_word implementation.
            items.append((4 + byte_value, char_start, char_end))
        char_position = char_end

    # The boundary token has no independent character span. If merged with word bytes,
    # the resulting token retains only the source characters occupied by those bytes.
    items.append((tokenizer.word_boundary_id, None, None))

    while len(items) > 1:
        best_position = None
        best_token_id = None
        for position in range(len(items) - 1):
            pair = (items[position][0], items[position + 1][0])
            merged_id = tokenizer.merge_to_id.get(pair)
            if merged_id is None:
                continue
            if best_token_id is None or merged_id < best_token_id:
                best_token_id = merged_id
                best_position = position
        if best_position is None:
            break
        position = best_position
        merged_item = _merge_two_offset_items(
            items[position], items[position + 1], best_token_id
        )
        items = items[:position] + [merged_item] + items[position + 2:]

    return items


def encode_with_offsets(text, tokenizer, add_bos=False, add_eos=False, verify_ids=True):
    """Return (token_ids, offsets), with offsets as Python-character [start, end) spans or None."""
    token_ids = []
    offsets = []

    if add_bos:
        token_ids.append(tokenizer.bos_token_id)
        offsets.append(None)

    matches = list(re.finditer(r"\S+", text, flags=re.UNICODE))
    regex_words = [match.group(0) for match in matches]
    if regex_words != text.split():
        raise RuntimeError("Whitespace matching differs from tokenizer.encode(text); refusing unsafe offsets.")

    for match in matches:
        word_items = _encode_word_with_offsets(match.group(0), match.start(), tokenizer)
        token_ids.extend(item[0] for item in word_items)
        offsets.extend(
            (item[1], item[2]) if item[1] is not None and item[2] is not None else None
            for item in word_items
        )

    if add_eos:
        token_ids.append(tokenizer.eos_token_id)
        offsets.append(None)

    if verify_ids:
        reference_ids = tokenizer.encode(text, add_bos=add_bos, add_eos=add_eos)
        if token_ids != reference_ids:
            raise AssertionError(
                "Offset-aware tokenization produced different token IDs from the original tokenizer."
            )
    if len(token_ids) != len(offsets):
        raise AssertionError("Token-ID/offset length mismatch.")
    return token_ids, offsets


OFFSET_TEST_STRINGS = [
    "The Supreme Court of India has jurisdiction over constitutional matters.",
    "  Case No. 123/2024\n\nCounsel for the appellant appeared.",
    "Unicode check: न्यायालय and café.",
    "tabs\tand multiple    spaces must preserve original character positions.",
]
for offset_test_text in OFFSET_TEST_STRINGS:
    test_ids, test_offsets = encode_with_offsets(offset_test_text, tokenizer)
    assert test_ids == tokenizer.encode(offset_test_text)
    assert len(test_ids) == len(test_offsets)
    for span in test_offsets:
        if span is not None:
            start, end = span
            assert 0 <= start < end <= len(offset_test_text)
print(f"Offset-aware tokenization checks passed for {len(OFFSET_TEST_STRINGS)} examples.")


## Audit exact-span alignment before choosing a prediction head

A token-aligned classifier can directly represent an entity boundary only when that boundary is present in the token offsets. This audit samples labelled documents and reports how often gold starts and ends align with token boundaries. It does not change annotations or decide the model architecture.

In [ ]:
def audit_span_alignment(document_ids, max_documents=30):
    rng = random.Random(SEED + 17)
    ids = list(document_ids)
    if len(ids) > max_documents:
        ids = rng.sample(ids, max_documents)

    total_entities = 0
    start_aligned = 0
    end_aligned = 0
    both_aligned = 0
    examples_not_aligned = []
    sampled_token_count = 0
    sampled_char_count = 0

    for document_id in ids:
        text = TEXTS_BY_ID[document_id]
        _, offsets = encode_with_offsets(text, tokenizer)
        valid_offsets = [span for span in offsets if span is not None]
        starts = {start for start, _ in valid_offsets}
        ends = {end for _, end in valid_offsets}
        sampled_token_count += len(offsets)
        sampled_char_count += len(text)

        for entity in ANNOTATIONS_BY_ID[document_id]:
            total_entities += 1
            start_ok = entity["start"] in starts
            end_ok = entity["end"] in ends
            start_aligned += int(start_ok)
            end_aligned += int(end_ok)
            both_aligned += int(start_ok and end_ok)
            if not (start_ok and end_ok) and len(examples_not_aligned) < 10:
                examples_not_aligned.append({
                    "document_id": document_id,
                    "gold_span": (entity["start"], entity["end"], entity["label"]),
                    "entity_text": text[entity["start"]:entity["end"]],
                    "start_aligned": start_ok,
                    "end_aligned": end_ok,
                })

    if total_entities == 0:
        raise ValueError("No entities found in the sampled documents; inspect the annotation structure.")

    report = {
        "sampled_documents": len(ids),
        "sampled_characters": sampled_char_count,
        "sampled_tokens": sampled_token_count,
        "entities_checked": total_entities,
        "start_boundary_alignment_rate": start_aligned / total_entities,
        "end_boundary_alignment_rate": end_aligned / total_entities,
        "both_boundary_alignment_rate": both_aligned / total_entities,
        "examples_not_aligned": examples_not_aligned,
    }
    return report


SPAN_ALIGNMENT_AUDIT = audit_span_alignment(TRAIN_DOC_IDS, max_documents=30)
print("Documents audited:", SPAN_ALIGNMENT_AUDIT["sampled_documents"])
print("Entities checked:", SPAN_ALIGNMENT_AUDIT["entities_checked"])
print("Start-boundary alignment rate:", f"{SPAN_ALIGNMENT_AUDIT['start_boundary_alignment_rate']:.2%}")
print("End-boundary alignment rate:  ", f"{SPAN_ALIGNMENT_AUDIT['end_boundary_alignment_rate']:.2%}")
print("Both-boundaries alignment rate:", f"{SPAN_ALIGNMENT_AUDIT['both_boundary_alignment_rate']:.2%}")
print("Examples requiring closer inspection:")
for example in SPAN_ALIGNMENT_AUDIT["examples_not_aligned"]:
    print(example)


## Phase 2 preprocessing: character spans to token labels

This section prepares labelled windows and reports spans that cannot be represented exactly by the existing tokenizer. It does **not** add an NER model head or a training algorithm. The notebook's existing token IDs and V1 checkpoint are left unchanged. Review the alignment counts before choosing a token-aligned prediction method.

In [ ]:
# Phase 2 label vocabulary and span-to-token alignment utilities.
# Unaligned entities receive approximate BIO targets.
# Conflicting token labels are masked using IGNORE_INDEX.

IGNORE_INDEX = -100

BIO_LABELS = (
    ["O"]
    + [f"B-{label}" for label in LABELS]
    + [f"I-{label}" for label in LABELS]
)

BIO_TO_ID = {name: idx for idx, name in enumerate(BIO_LABELS)}
ID_TO_BIO = {idx: name for name, idx in BIO_TO_ID.items()}
O_LABEL_ID = BIO_TO_ID["O"]


def _overlapping_token_indices(entity, offsets):
    start, end = entity["start"], entity["end"]

    return [
        i
        for i, span in enumerate(offsets)
        if span is not None and span[0] < end and span[1] > start
    ]


def make_bio_targets(document_id, token_offsets, entities):
    """
    Create BIO targets for exact and non-exact entity boundaries.

    Exact spans:
        Use the original exact token boundaries.

    Non-exact spans:
        Approximate using all valid tokens overlapping the gold span.

    Conflicting token labels:
        Mask conflicting positions with IGNORE_INDEX.
    """
    token_label_ids = [
        O_LABEL_ID if span is not None else IGNORE_INDEX
        for span in token_offsets
    ]

    proposed_labels = {}
    ignored_token_indices = set()

    report = {
        "document_id": str(document_id),
        "entity_count": len(entities),
        "exactly_aligned_entities": 0,
        "unaligned_entities": 0,
        "entities_without_overlapping_tokens": 0,
        "overlapping_entity_conflicts": 0,
        "approximated_entities": 0,
        "unaligned_examples": [],
    }

    for entity in entities:
        start = entity["start"]
        end = entity["end"]
        label = entity["label"]

        if label not in LABELS:
            raise ValueError(f"Unknown entity label: {label}")

        if not (0 <= start < end):
            raise ValueError(
                f"Invalid entity span for document {document_id}: "
                f"({start}, {end}, {label})"
            )

        overlapping = _overlapping_token_indices(entity, token_offsets)

        if not overlapping:
            report["entities_without_overlapping_tokens"] += 1
            report["unaligned_entities"] += 1

            if len(report["unaligned_examples"]) < 5:
                report["unaligned_examples"].append(
                    (start, end, label, "no_token_overlap")
                )
            continue

        first_index = overlapping[0]
        last_index = overlapping[-1]

        exact_alignment = (
            token_offsets[first_index][0] == start
            and token_offsets[last_index][1] == end
        )

        if exact_alignment:
            report["exactly_aligned_entities"] += 1
        else:
            report["unaligned_entities"] += 1
            report["approximated_entities"] += 1

            if len(report["unaligned_examples"]) < 5:
                report["unaligned_examples"].append(
                    (start, end, label, "approximated")
                )

        target_ids = (
            [BIO_TO_ID[f"B-{label}"]]
            + [BIO_TO_ID[f"I-{label}"]] * (len(overlapping) - 1)
        )

        proposal = {
            idx: target_id
            for idx, target_id in zip(overlapping, target_ids)
        }

        # Check conflicts before committing this entity's labels.
        conflict_indices = [
            idx
            for idx, target_id in proposal.items()
            if idx in proposed_labels
            and proposed_labels[idx] != target_id
        ]

        if conflict_indices:
            report["overlapping_entity_conflicts"] += 1

            # Mask all tokens participating in the conflicting annotations.
            ignored_token_indices.update(proposal)
            ignored_token_indices.update(
                idx for idx in proposal if idx in proposed_labels
            )
            ignored_token_indices.update(conflict_indices)

            continue

        # Do not allow later entities to overwrite tokens already
        # identified as ambiguous.
        for idx, target_id in proposal.items():
            if idx not in ignored_token_indices:
                proposed_labels[idx] = target_id

    for idx, target_id in proposed_labels.items():
        if idx not in ignored_token_indices:
            token_label_ids[idx] = target_id

    for idx in ignored_token_indices:
        token_label_ids[idx] = IGNORE_INDEX

    return token_label_ids, report


def prepare_lner_windows(
    document_id,
    text,
    entities,
    max_length,
    window_overlap=0,
    tokenizer=None,
):
    """
    Tokenize a document and create overlapping windows.

    Offsets refer to the original document.
    """
    if max_length <= 0:
        raise ValueError("max_length must be positive.")

    if not 0 <= window_overlap < max_length:
        raise ValueError(
            "window_overlap must satisfy 0 <= window_overlap < max_length."
        )

    active_tokenizer = (
        tokenizer if tokenizer is not None else globals()["tokenizer"]
    )

    token_ids, token_offsets = encode_with_offsets(text, active_tokenizer)

    token_targets, alignment_report = make_bio_targets(
        document_id,
        token_offsets,
        entities,
    )

    if not (
        len(token_ids) == len(token_targets) == len(token_offsets)
    ):
        raise AssertionError(
            "Token IDs, labels, and offsets must have identical lengths."
        )

    windows = []

    if not token_ids:
        return windows, alignment_report

    step = max_length - window_overlap
    token_start = 0

    while token_start < len(token_ids):
        token_end = min(token_start + max_length, len(token_ids))

        windows.append({
            "document_id": str(document_id),
            "token_start": token_start,
            "input_ids": token_ids[token_start:token_end],
            "labels": token_targets[token_start:token_end],
            "offsets": token_offsets[token_start:token_end],
        })

        if token_end == len(token_ids):
            break

        token_start += step

    return windows, alignment_report


print(
    f"BIO vocabulary size: {len(BIO_LABELS)} "
    f"(O plus B/I for {len(LABELS)} entity types)"
)
print("Special/unlabelled token positions use IGNORE_INDEX =", IGNORE_INDEX)

In [ ]:
# Verify the updated target builder on the existing training documents.

sample_ids = TRAIN_DOC_IDS[:30]
totals = {
    "entities": 0,
    "exactly_aligned": 0,
    "approximated": 0,
    "without_tokens": 0,
    "conflicts": 0,
}

for document_id in sample_ids:
    text = TEXTS_BY_ID[str(document_id)]
    entities = ANNOTATIONS_BY_ID[str(document_id)]

    _, offsets = encode_with_offsets(text, tokenizer)
    _, report = make_bio_targets(str(document_id), offsets, entities)

    totals["entities"] += report["entity_count"]
    totals["exactly_aligned"] += report["exactly_aligned_entities"]
    totals["approximated"] += report["approximated_entities"]
    totals["without_tokens"] += report["entities_without_overlapping_tokens"]
    totals["conflicts"] += report["overlapping_entity_conflicts"]

print("Updated target-builder audit")
print("=" * 40)
for key, value in totals.items():
    print(f"{key}: {value}")

print(
    "\nEntities receiving exact or approximate supervision:",
    totals["exactly_aligned"] + totals["approximated"],
    "of",
    totals["entities"],
)

## Window-level dataset objects for Phase 2

The dataset class below uses the preprocessing helpers above to build token/label windows on demand when explicitly constructed. It stores token IDs and label IDs as tensors, not the full character-offset objects. The custom collator pads on the right and masks padded targets with `IGNORE_INDEX`. Because this will later feed a token-classification head, the model/training implementation must use the labels only at valid positions.

In [ ]:
from torch.utils.data import Dataset, DataLoader


class LNERWindowDataset(Dataset):
    """Materialize document windows for a selected set of document IDs."""
    def __init__(
        self,
        document_ids,
        text_by_id,
        entities_by_id,
        max_length,
        window_overlap_tokens,
        tokenizer,
    ):
        self.samples = []
        self.alignment_reports = []
        self.document_count = 0
        self.token_count = 0

        for document_id in document_ids:
            doc_id = str(document_id)
            text = text_by_id[doc_id]
            entities = entities_by_id[doc_id]
            windows, report = prepare_lner_windows(
                document_id=doc_id,
                text=text,
                entities=entities,
                max_length=max_length,
                window_overlap=window_overlap_tokens,
                tokenizer=tokenizer,
            )
            self.alignment_reports.append(report)
            self.document_count += 1
            for window in windows:
                input_ids = torch.tensor(window["input_ids"], dtype=torch.long)
                labels = torch.tensor(window["labels"], dtype=torch.long)
                if input_ids.numel() != labels.numel():
                    raise AssertionError(f"Input/label length mismatch for {doc_id}.")
                self.token_count += int(input_ids.numel())
                self.samples.append({
                    "input_ids": input_ids,
                    "labels": labels,
                    "document_id": doc_id,
                    "token_start": int(window["token_start"]),
                })

        self.alignment_summary = {
            "documents": self.document_count,
            "windows": len(self.samples),
            "window_tokens_including_overlap": self.token_count,
            "entities": sum(x["entity_count"] for x in self.alignment_reports),
            "exactly_aligned_entities": sum(x["exactly_aligned_entities"] for x in self.alignment_reports),
            "unaligned_entities": sum(x["unaligned_entities"] for x in self.alignment_reports),
            "overlapping_entity_conflicts": sum(x["overlapping_entity_conflicts"] for x in self.alignment_reports),
        }
        total_entities = self.alignment_summary["entities"]
        self.alignment_summary["exact_alignment_rate"] = (
            self.alignment_summary["exactly_aligned_entities"] / total_entities
            if total_entities else 0.0
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


def collate_lner_windows(batch, pad_token_id):
    """Right-pad input IDs and mask pad targets; retain IDs for document-level decoding."""
    if not batch:
        raise ValueError("Cannot collate an empty batch.")
    max_len = max(item["input_ids"].numel() for item in batch)
    input_ids = torch.full(
        (len(batch), max_len), int(pad_token_id), dtype=torch.long
    )
    labels = torch.full(
        (len(batch), max_len), IGNORE_INDEX, dtype=torch.long
    )
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)

    for row, item in enumerate(batch):
        length = item["input_ids"].numel()
        input_ids[row, :length] = item["input_ids"]
        labels[row, :length] = item["labels"]
        attention_mask[row, :length] = 1

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask,
        "document_ids": [item["document_id"] for item in batch],
        "token_starts": [item["token_start"] for item in batch],
    }


def build_phase2_window_datasets(window_overlap_tokens):
    """Explicitly build train/validation windows after the author selects overlap."""
    train_dataset = LNERWindowDataset(
        document_ids=TRAIN_DOC_IDS,
        text_by_id=TEXTS_BY_ID,
        entities_by_id=ANNOTATIONS_BY_ID,
        max_length=MODEL_CONTEXT_LENGTH,
        window_overlap_tokens=window_overlap_tokens,
        tokenizer=tokenizer,
    )
    validation_dataset = LNERWindowDataset(
        document_ids=VAL_DOC_IDS,
        text_by_id=TEXTS_BY_ID,
        entities_by_id=ANNOTATIONS_BY_ID,
        max_length=MODEL_CONTEXT_LENGTH,
        window_overlap_tokens=window_overlap_tokens,
        tokenizer=tokenizer,
    )
    return train_dataset, validation_dataset


print("LNERWindowDataset, collate_lner_windows, and build_phase2_window_datasets are defined.")
print("The complete train/validation tensors are NOT built until build_phase2_window_datasets(...) is called.")


## Sanity-check the Phase 2 feature builder

This previews a few labelled documents only. It does not tokenize the complete dataset in advance or start training. The output reports the number of windows and the number of gold entities whose boundaries match the tokenizer's character offsets exactly.

In [ ]:
# Small preprocessing smoke test; no model training occurs in this cell.
PHASE2_PREVIEW_DOCUMENTS = min(3, len(TRAIN_DOC_IDS))
PREPROCESSING_PREVIEW = []

for document_id in TRAIN_DOC_IDS[:PHASE2_PREVIEW_DOCUMENTS]:
    text = TEXTS_BY_ID[document_id]
    entities = ANNOTATIONS_BY_ID[document_id]
    windows, report = prepare_lner_windows(
        document_id=document_id,
        text=text,
        entities=entities,
        max_length=MODEL_CONTEXT_LENGTH,
        window_overlap=0,
    )
    PREPROCESSING_PREVIEW.append({"windows": windows, "alignment": report})
    print({
        "document_id": str(document_id),
        "characters": len(text),
        "tokens": sum(len(window["input_ids"]) for window in windows),
        "windows": len(windows),
        "entities": report["entity_count"],
        "exactly_aligned_entities": report["exactly_aligned_entities"],
        "unaligned_entities": report["unaligned_entities"],
        "overlapping_entity_conflicts": report["overlapping_entity_conflicts"],
    })

# Keep only summary information in the notebook namespace after printing the sample.
PREPROCESSING_PREVIEW_SUMMARY = [item["alignment"] for item in PREPROCESSING_PREVIEW]
del PREPROCESSING_PREVIEW


## Local strict-span evaluation and submission utilities

These helpers compare exact `(start, end, label)` triples and create the required JSON/ZIP once a model has produced predictions. They do not train or choose a model. For final reporting, compare this local calculation with the official evaluator provided for the capstone.

In [ ]:
from collections import Counter


def _prediction_entity_triples(raw_entities):
    """Normalize prediction entities into (start, end, label) tuples."""
    triples = []
    for entity in raw_entities:
        if isinstance(entity, dict):
            start = entity.get("start", entity.get("start_char", entity.get("start_offset")))
            end = entity.get("end", entity.get("end_char", entity.get("end_offset")))
            label = entity.get("label", entity.get("entity_label", entity.get("type")))
        elif isinstance(entity, (list, tuple)) and len(entity) >= 3:
            start, end, label = entity[:3]
        else:
            raise TypeError(f"Unsupported predicted entity representation: {entity!r}")
        triples.append((int(start), int(end), str(label)))
    return triples


def strict_span_metrics(gold_by_id, predicted_by_id, labels=None, document_ids=None):
    """Compute strict exact-span per-label precision/recall/F1 and macro F1.

    Inputs can use the normalized gold dictionaries already created above and predictions
    in [start, end, label] form. This is a local diagnostic, not a replacement for the
    official capstone evaluator.
    """
    labels = list(labels if labels is not None else LABELS)
    ids = list(document_ids) if document_ids is not None else sorted(
        set(map(str, gold_by_id)) | set(map(str, predicted_by_id))
    )
    gold_counts = {label: Counter() for label in labels}
    pred_counts = {label: Counter() for label in labels}

    for document_id in ids:
        doc_id = str(document_id)
        for entity in gold_by_id.get(doc_id, gold_by_id.get(document_id, [])):
            if isinstance(entity, dict):
                triple = (int(entity["start"]), int(entity["end"]), str(entity["label"]))
            else:
                triple = _prediction_entity_triples([entity])[0]
            if triple[2] in gold_counts:
                gold_counts[triple[2]][(doc_id, triple[0], triple[1], triple[2])] += 1

        for entity in predicted_by_id.get(doc_id, predicted_by_id.get(document_id, [])):
            triple = _prediction_entity_triples([entity])[0]
            if triple[2] in pred_counts:
                pred_counts[triple[2]][(doc_id, triple[0], triple[1], triple[2])] += 1

    per_label = {}
    f1_values = []
    for label in labels:
        tp = sum((gold_counts[label] & pred_counts[label]).values())
        fp = sum(pred_counts[label].values()) - tp
        fn = sum(gold_counts[label].values()) - tp
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if precision + recall else 0.0
        per_label[label] = {
            "tp": tp, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1,
        }
        f1_values.append(f1)

    return {
        "strict_macro_f1": sum(f1_values) / len(f1_values) if f1_values else 0.0,
        "per_label": per_label,
        "documents_evaluated": len(ids),
    }


def write_lner_submission(predictions_by_id, prediction_path=None, zip_path=None):
    """Validate and write Phase 2 prediction.json and a ZIP with the JSON at its root."""
    prediction_path = prediction_path or PHASE2_PREDICTION_PATH
    zip_path = zip_path or PHASE2_SUBMISSION_ZIP
    normalized = {str(key): _prediction_entity_triples(value) for key, value in predictions_by_id.items()}
    expected_ids = set(map(str, TEST_TEXTS))
    received_ids = set(normalized)
    if expected_ids != received_ids:
        raise ValueError(
            "Prediction document IDs must exactly match fold3-text.json. "
            f"Missing={len(expected_ids - received_ids)}, extra={len(received_ids - expected_ids)}."
        )

    for document_id, triples in normalized.items():
        text = TEST_TEXTS[document_id]
        for start, end, label in triples:
            if label not in LABELS:
                raise ValueError(f"Unknown label {label!r} for document {document_id}.")
            if not (0 <= start < end <= len(text)):
                raise ValueError(
                    f"Invalid span {(start, end, label)} for document {document_id} with text length {len(text)}."
                )

    prediction_dir = os.path.dirname(prediction_path)
    zip_dir = os.path.dirname(zip_path)
    if prediction_dir:
        os.makedirs(prediction_dir, exist_ok=True)
    if zip_dir:
        os.makedirs(zip_dir, exist_ok=True)
    with open(prediction_path, "w", encoding="utf-8") as handle:
        json.dump(normalized, handle, ensure_ascii=False)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        archive.write(prediction_path, arcname="prediction.json")

    print("Saved predictions:", prediction_path)
    print("Saved submission ZIP:", zip_path)
    with zipfile.ZipFile(zip_path, "r") as archive:
        print("ZIP members:", archive.namelist())


In [ ]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Parameters in millions: {total_params / 1e6:.2f}M")

assert 50_000_000 <= total_params <= 300_000_000

print("\nParameter count check passed.")

In [ ]:
# model.eval()

# test_batch = next(iter(train_loader))

# test_input_ids = test_batch["input_ids"][:2].to(DEVICE)
# test_labels = test_batch["labels"][:2].to(DEVICE)

# with torch.no_grad():
#     test_logits, _ = model(
#         test_input_ids,
#         use_cache=False
#     )

# print("Input shape:", test_input_ids.shape)
# print("Labels shape:", test_labels.shape)
# print("Logits shape:", test_logits.shape)

# expected_shape = (
#     test_input_ids.shape[0],
#     test_input_ids.shape[1],
#     tokenizer.vocab_size
# )

# assert test_logits.shape == expected_shape
# assert torch.isfinite(test_logits).all()

# print("\nForward pass check passed.")

In [ ]:
# model.train()

# logits, _ = model(
#     test_input_ids,
#     use_cache=False
# )

# loss = F.cross_entropy(
#     logits.reshape(-1, tokenizer.vocab_size),
#     test_labels.reshape(-1)
# )

# print(f"Initial loss: {loss.item():.6f}")

# assert torch.isfinite(loss), "Loss is not finite."

# # Clear any existing gradients before this test.
# model.zero_grad(set_to_none=True)

# loss.backward()

# # Check that gradients were actually produced.
# parameters_with_grad = 0
# total_grad_norm_sq = 0.0

# for parameter in model.parameters():
#     if parameter.grad is not None:
#         parameters_with_grad += 1
#         total_grad_norm_sq += (
#             parameter.grad.detach()
#             .float()
#             .norm(2)
#             .item() ** 2
#         )

# grad_norm = total_grad_norm_sq ** 0.5

# print("Parameters with gradients:", parameters_with_grad)
# print(f"Gradient norm: {grad_norm:.6f}")

# assert parameters_with_grad > 0
# assert math.isfinite(grad_norm)

# print("\nLoss and backward-pass check passed.")

In [ ]:
#Causal Mask Test
# A decoder-only model must not allow a token to use
# information from future positions.We test this by changing a future token and checking that an earlier position gives the same output.

model.eval()

torch.manual_seed(SEED)

test_ids = torch.randint(
    low=0,
    high=tokenizer.vocab_size,
    size=(1, 16),
    device=DEVICE
)

# Make a copy and change only a future token.
modified_ids = test_ids.clone()
modified_ids[0, 10] = (
    modified_ids[0, 10] + 1
) % tokenizer.vocab_size

with torch.no_grad():
    logits_original, _ = model(
        test_ids,
        use_cache=False
    )

    logits_modified, _ = model(
        modified_ids,
        use_cache=False
    )

# Position 5 cannot see position 10.
earlier_difference = (
    logits_original[0, 5]
    - logits_modified[0, 5]
).abs().max().item()

# Position 10 can see itself, so changing it should affect
# its output in general.
current_difference = (
    logits_original[0, 10]
    - logits_modified[0, 10]
).abs().max().item()

print(
    f"Difference at earlier position: {earlier_difference:.10f}"
)

print(
    f"Difference at modified position: {current_difference:.10f}"
)

assert earlier_difference < 1e-5
assert current_difference > 1e-6

print("\nCausal mask check passed.")

In [ ]:
# KV Cache Equivalence Test
# During generation, KV caching should give the same results
# as processing the complete sequence normally.
# We compare:
# 1. Full-sequence forward pass
# 2. Token-by-token forward pass using the KV cache

model.eval()

torch.manual_seed(SEED)

cache_test_ids = torch.randint(
    low=0,
    high=tokenizer.vocab_size,
    size=(1, 16),
    device=DEVICE
)

with torch.no_grad():

    # Full sequence in one forward pass.
    full_logits, _ = model(
        cache_test_ids,
        use_cache=False
    )

    # Process one token at a time while keeping the
    # previous keys and values in the cache.
    past_key_values = None
    cached_logits = []

    for position in range(
        cache_test_ids.size(1)
    ):

        current_token = cache_test_ids[
            :, position:position + 1
        ]

        logits, past_key_values = model(
            current_token,
            past_key_values=past_key_values,
            use_cache=True
        )

        cached_logits.append(logits)

    cached_logits = torch.cat(
        cached_logits,
        dim=1
    )


# Compare full-sequence and cached outputs.
max_difference = (
    full_logits - cached_logits
).abs().max().item()

mean_difference = (
    full_logits - cached_logits
).abs().mean().item()

print(
    f"Maximum logit difference: {max_difference:.10f}"
)

print(
    f"Mean logit difference:    {mean_difference:.10f}"
)

assert torch.allclose(
    full_logits,
    cached_logits,
    atol=1e-4,
    rtol=1e-4
)

print("\nKV cache equivalence check passed.")

In [ ]:
# # Tiny Overfit Test
# # Sanity check

# model.train()

# tiny_input_ids = test_batch["input_ids"][:2].to(DEVICE)
# tiny_labels = test_batch["labels"][:2].to(DEVICE)

# tiny_optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=1e-3,
#     weight_decay=0.0
# )

# initial_tiny_loss = None

# for step in range(100):

#     tiny_optimizer.zero_grad(
#         set_to_none=True
#     )

#     logits, _ = model(
#         tiny_input_ids,
#         use_cache=False
#     )

#     loss = F.cross_entropy(
#         logits.reshape(-1, tokenizer.vocab_size),
#         tiny_labels.reshape(-1)
#     )

#     if initial_tiny_loss is None:
#         initial_tiny_loss = loss.item()

#     loss.backward()

#     torch.nn.utils.clip_grad_norm_(
#         model.parameters(),
#         max_norm=1.0
#     )

#     tiny_optimizer.step()

#     if step == 0 or (step + 1) % 20 == 0:
#         print(
#             f"Step {step + 1:3d} | "
#             f"Loss: {loss.item():.6f}"
#         )

# final_tiny_loss = loss.item()

# print(
#     f"\nInitial tiny loss: {initial_tiny_loss:.6f}"
# )

# print(
#     f"Final tiny loss:   {final_tiny_loss:.6f}"
# )

# assert final_tiny_loss < initial_tiny_loss

# print("\nTiny overfit check passed.")

In [ ]:
# # Fresh V1 model for the actual training run.

# model = V1LegalLM(
#     vocab_size=tokenizer.vocab_size,
#     n_embd=N_EMBD,
#     n_layer=N_LAYER,
#     n_head=N_HEAD,
#     n_kv_head=N_KV_HEAD,
#     max_seq_len=MODEL_CONTEXT_LENGTH,
#     ffn_multiplier=FFN_MULTIPLIER,
#     dropout=DROPOUT
# ).to(DEVICE)

# print("Fresh V1 model created for training.")
# print(
#     f"Parameters: "
#     f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M"
# )

## 10. Training Arguments

In [ ]:
# import os
# import json
# import math
# import time

# # Fresh optimizer for the freshly initialized model.
# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=LEARNING_RATE,
#     betas=(0.9, 0.95),
#     weight_decay=WEIGHT_DECAY
# )

# # Number of tokens processed in one optimizer step.
# TOKENS_PER_STEP = (
#     PER_DEVICE_BATCH_SIZE
#     * GRADIENT_ACCUMULATION_STEPS
#     * TRAIN_BLOCK_SIZE
# )

# MAX_TRAIN_STEPS = math.ceil(
#     TRAIN_TOKENS / TOKENS_PER_STEP
# )

# print(f"Tokens per optimizer step: {TOKENS_PER_STEP:,}")
# print(f"Total training tokens: {TRAIN_TOKENS:,}")
# print(f"Planned optimizer steps: {MAX_TRAIN_STEPS:,}")

# # Cosine learning-rate schedule with warmup.
# def get_learning_rate(step):
#     if step < WARMUP_STEPS:
#         return LEARNING_RATE * (
#             (step + 1) / WARMUP_STEPS
#         )

#     progress = (
#         step - WARMUP_STEPS
#     ) / max(
#         1,
#         MAX_TRAIN_STEPS - WARMUP_STEPS
#     )

#     progress = min(max(progress, 0.0), 1.0)

#     return LEARNING_RATE * 0.5 * (
#         1.0 + math.cos(math.pi * progress)
#     )

# print("Training setup ready.")

In [ ]:
# # Checkpoint Setup

# CHECKPOINT_DIR = os.path.join(
#     OUTPUT_DIR,
#     "checkpoints"
# )

# os.makedirs(
#     CHECKPOINT_DIR,
#     exist_ok=True
# )

# LATEST_CHECKPOINT = os.path.join(
#     CHECKPOINT_DIR,
#     "latest.pt"
# )

# BEST_CHECKPOINT = os.path.join(
#     CHECKPOINT_DIR,
#     "best.pt"
# )

# history = {
#     "step": [],
#     "train_loss": [],
#     "val_loss": [],
#     "val_perplexity": [],
#     "learning_rate": [],
#     "tokens_seen": []
# }

# best_val_loss = float("inf")

# print("Checkpoint directory:", CHECKPOINT_DIR)

In [ ]:
# # ---------------------------------------------------------
# # 12. Validation
# # ---------------------------------------------------------
# # Evaluate the model without updating its weights.
# # ---------------------------------------------------------

# @torch.no_grad()
# def evaluate_model():
#     model.eval()

#     total_loss = 0.0
#     total_tokens = 0

#     for batch in val_loader:

#         input_ids = batch["input_ids"].to(
#             DEVICE,
#             non_blocking=True
#         )

#         labels = batch["labels"].to(
#             DEVICE,
#             non_blocking=True
#         )

#         logits, _ = model(
#             input_ids,
#             use_cache=False
#         )

#         loss = F.cross_entropy(
#             logits.reshape(
#                 -1,
#                 tokenizer.vocab_size
#             ),
#             labels.reshape(-1)
#         )

#         tokens = labels.numel()

#         total_loss += (
#             loss.item() * tokens
#         )

#         total_tokens += tokens

#     val_loss = (
#         total_loss / total_tokens
#     )

#     val_perplexity = math.exp(
#         min(val_loss, 20)
#     )

#     model.train()

#     return val_loss, val_perplexity

# print("Validation function ready.")

## 11. Training

In [ ]:
# # V1 training loop

# model.train()

# global_step = 0
# tokens_seen = 0
# start_time = time.time()

# optimizer.zero_grad(set_to_none=True)

# train_iterator = iter(train_loader)

# while global_step < MAX_TRAIN_STEPS:

#     accumulated_loss = 0.0

#     for micro_step in range(GRADIENT_ACCUMULATION_STEPS):

#         batch = next(train_iterator)

#         input_ids = batch["input_ids"].to(
#             DEVICE,
#             non_blocking=True
#         )

#         labels = batch["labels"].to(
#             DEVICE,
#             non_blocking=True
#         )

#         logits, _ = model(
#             input_ids,
#             use_cache=False
#         )

#         loss = F.cross_entropy(
#             logits.reshape(-1, tokenizer.vocab_size),
#             labels.reshape(-1)
#         )

#         accumulated_loss += loss.item()

#         loss = loss / GRADIENT_ACCUMULATION_STEPS
#         loss.backward()

#     current_lr = get_learning_rate(global_step)

#     for param_group in optimizer.param_groups:
#         param_group["lr"] = current_lr

#     grad_norm = torch.nn.utils.clip_grad_norm_(
#         model.parameters(),
#         MAX_GRAD_NORM
#     )

#     optimizer.step()
#     optimizer.zero_grad(set_to_none=True)

#     global_step += 1
#     tokens_seen += TOKENS_PER_STEP

#     train_loss = (
#         accumulated_loss / GRADIENT_ACCUMULATION_STEPS
#     )

#     if global_step == 1 or global_step % LOG_EVERY_STEPS == 0:

#         elapsed = time.time() - start_time
#         tokens_per_second = tokens_seen / max(elapsed, 1e-6)

#         print(
#             f"Step {global_step:,}/{MAX_TRAIN_STEPS:,} | "
#             f"Loss: {train_loss:.4f} | "
#             f"LR: {current_lr:.2e} | "
#             f"Grad: {grad_norm.item():.3f} | "
#             f"Tokens: {tokens_seen:,} | "
#             f"Tok/s: {tokens_per_second:,.0f}"
#         )

#         history["step"].append(global_step)
#         history["train_loss"].append(train_loss)
#         history["learning_rate"].append(current_lr)
#         history["tokens_seen"].append(tokens_seen)

#     if (
#         global_step % EVAL_EVERY_STEPS == 0
#         or global_step == MAX_TRAIN_STEPS
#     ):

#         val_loss, val_ppl = evaluate_model()

#         print(
#             f"\nValidation | "
#             f"Step {global_step:,} | "
#             f"Loss: {val_loss:.4f} | "
#             f"Perplexity: {val_ppl:.2f}\n"
#         )

#         history["val_loss"].append(val_loss)
#         history["val_perplexity"].append(val_ppl)

#         if val_loss < best_val_loss:

#             best_val_loss = val_loss

#             torch.save(
#                 {
#                     "step": global_step,
#                     "model_state_dict": model.state_dict(),
#                     "optimizer_state_dict": optimizer.state_dict(),
#                     "best_val_loss": best_val_loss,
#                     "history": history
#                 },
#                 BEST_CHECKPOINT
#             )

#             print("New best checkpoint saved.")

#     if (
#         global_step % SAVE_EVERY_STEPS == 0
#         or global_step == MAX_TRAIN_STEPS
#     ):

#         torch.save(
#             {
#                 "step": global_step,
#                 "model_state_dict": model.state_dict(),
#                 "optimizer_state_dict": optimizer.state_dict(),
#                 "best_val_loss": best_val_loss,
#                 "history": history
#             },
#             LATEST_CHECKPOINT
#         )

#         with open(
#             os.path.join(
#                 OUTPUT_DIR,
#                 "training_history.json"
#             ),
#             "w",
#             encoding="utf-8"
#         ) as f:
#             json.dump(
#                 history,
#                 f,
#                 indent=2
#             )

#         print("Latest checkpoint saved.")

# print("\nV1 training complete.")
# print(f"Best validation loss: {best_val_loss:.6f}")

In [ ]:
# # Load the checkpoint with the best validation loss

# best_checkpoint = torch.load(
#     BEST_CHECKPOINT,
#     map_location=DEVICE
# )

# model.load_state_dict(
#     best_checkpoint["model_state_dict"]
# )

# model.eval()

# best_val_loss = best_checkpoint["best_val_loss"]

# print("Best V1 checkpoint loaded.")
# print(f"Best validation loss: {best_val_loss:.6f}")
# print(f"Best checkpoint step: {best_checkpoint['step']:,}")
# print(f"Best validation perplexity: {math.exp(best_val_loss):.4f}")

## 12. Perplexity

In [ ]:
# # Perplexity from the best validation checkpoint.

# perplexity = math.exp(best_val_loss)

# print(f"Validation loss: {best_val_loss:.6f}")
# print(f"Validation perplexity: {perplexity:.4f}")

## 13. Plot Training and Validation Loss

In [ ]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(10, 5))

# plt.plot(
#     history["step"],
#     history["train_loss"],
#     label="Training Loss"
# )

# val_steps = [
#     min(
#         (i + 1) * EVAL_EVERY_STEPS,
#         MAX_TRAIN_STEPS
#     )
#     for i in range(
#         len(history["val_loss"])
#     )
# ]

# plt.plot(
#     val_steps,
#     history["val_loss"],
#     marker="o",
#     label="Validation Loss"
# )

# plt.xlabel("Training Step")
# plt.ylabel("Loss")
# plt.title("V1 Training and Validation Loss")
# plt.legend()
# plt.grid(True)
# plt.tight_layout()

# LOSS_PLOT_PATH = os.path.join(
#     OUTPUT_DIR,
#     "v1_loss_curve.png"
# )

# plt.savefig(
#     LOSS_PLOT_PATH,
#     dpi=200,
#     bbox_inches="tight"
# )

# plt.show()

# print("Saved:", LOSS_PLOT_PATH)

## 14. Save Final Model to Kaggle Output

In [ ]:
# import os
# import json

# os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

# final_model_path = os.path.join(
#     MODEL_OUTPUT_DIR,
#     "v1_model.pt"
# )

# torch.save(
#     {
#         "model_state_dict": model.state_dict(),
#         "config": {
#             "vocab_size": tokenizer.vocab_size,
#             "n_embd": N_EMBD,
#             "n_layer": N_LAYER,
#             "n_head": N_HEAD,
#             "n_kv_head": N_KV_HEAD,
#             "ffn_multiplier": FFN_MULTIPLIER,
#             "dropout": DROPOUT,
#             "context_length": MODEL_CONTEXT_LENGTH
#         },
#         "best_val_loss": best_val_loss,
#         "best_val_perplexity": math.exp(best_val_loss)
#     },
#     final_model_path
# )

# print("Final V1 model saved to:")
# print(final_model_path)

## 15. Push Model to Hugging Face Hub

In [ ]:
# model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)

# print("Model and tokenizer pushed successfully.")

## 16. Load Test Data from HuggingFace

In [ ]:
# test_file_path = hf_hub_download(
#     repo_id=HF_DATASET_REPO,
#     filename=TEST_FILE,
#     repo_type="dataset",
#     token=HF_TOKEN
# )

# print("Test file:", test_file_path)

## 17. Load Test Prompts

In [ ]:
# with open(test_file_path, "r", encoding="utf-8") as f:
#     test_data = json.load(f)

# print("Number of test examples:", len(test_data))

## 18. Load Final Model for Generation

In [ ]:
# # Load the best V1 model for test generation.

# generation_model = V1LegalLM(
#     vocab_size=tokenizer.vocab_size,
#     n_embd=N_EMBD,
#     n_layer=N_LAYER,
#     n_head=N_HEAD,
#     n_kv_head=N_KV_HEAD,
#     max_seq_len=MODEL_CONTEXT_LENGTH,
#     ffn_multiplier=FFN_MULTIPLIER,
#     dropout=DROPOUT
# ).to(DEVICE)

# generation_model.load_state_dict(
#     model.state_dict()
# )

# generation_model.eval()

# generation_tokenizer = tokenizer

# print("V1 generation model loaded.")
# print("Device:", DEVICE)
# print(
#     f"Parameters: "
#     f"{sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M"
# )
# print("KV cache is supported for generation.")

## 19. Generate 128 Words
#### (words, not tokens)

In [ ]:
# # Generate legal-text continuations using the V1 model and KV cache.

# def apply_repetition_penalty(logits, generated_tokens, penalty):
#     if penalty == 1.0 or len(generated_tokens) == 0:
#         return logits

#     for token_id in set(generated_tokens):
#         if logits[token_id] < 0:
#             logits[token_id] *= penalty
#         else:
#             logits[token_id] /= penalty

#     return logits


# def sample_next_token(logits):
#     logits = logits / max(TEMPERATURE, 1e-6)

#     logits[PAD_TOKEN_ID] = float("-inf")
#     logits[BOS_TOKEN_ID] = float("-inf")

#     if TOP_K > 0:
#         k = min(TOP_K, logits.numel())

#         top_values, _ = torch.topk(
#             logits,
#             k
#         )

#         cutoff = top_values[-1]
#         logits[logits < cutoff] = float("-inf")

#     if TOP_P < 1.0:
#         sorted_logits, sorted_indices = torch.sort(
#             logits,
#             descending=True
#         )

#         sorted_probs = torch.softmax(
#             sorted_logits,
#             dim=-1
#         )

#         cumulative_probs = torch.cumsum(
#             sorted_probs,
#             dim=-1
#         )

#         remove = cumulative_probs > TOP_P

#         remove[1:] = remove[:-1].clone()
#         remove[0] = False

#         sorted_logits[remove] = float("-inf")

#         logits = torch.full_like(
#             logits,
#             float("-inf")
#         )

#         logits.scatter_(
#             0,
#             sorted_indices,
#             sorted_logits
#         )

#     probabilities = torch.softmax(
#         logits,
#         dim=-1
#     )

#     return torch.multinomial(
#         probabilities,
#         num_samples=1
#     ).item()


# def generate_one(
#     prompt,
#     model,
#     tokenizer,
#     device,
#     target_words=128
# ):
#     token_ids = tokenizer.encode(prompt)

#     max_prompt_tokens = (
#         MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS
#     )

#     # Keep the most recent part of the prompt if it is too long.
#     if len(token_ids) > max_prompt_tokens:
#         token_ids = token_ids[-max_prompt_tokens:]

#     input_ids = torch.tensor(
#         [token_ids],
#         dtype=torch.long,
#         device=device
#     )

#     generated_tokens = []

#     with torch.inference_mode():

#         # First pass processes the complete prompt and builds the KV cache.
#         logits, past_key_values = model(
#             input_ids,
#             use_cache=True
#         )

#         next_logits = logits[:, -1, :][0]

#         while len(generated_tokens) < MAX_GENERATION_TOKENS:

#             next_logits = apply_repetition_penalty(
#                 next_logits,
#                 generated_tokens,
#                 REPETITION_PENALTY
#             )

#             next_token = sample_next_token(
#                 next_logits.clone()
#             )

#             generated_tokens.append(
#                 next_token
#             )

#             generated_text = tokenizer.decode(
#                 generated_tokens
#             )

#             if len(generated_text.split()) >= target_words:
#                 break

#             # Do not exceed the model context window.
#             current_length = (
#                 len(token_ids)
#                 + len(generated_tokens)
#             )

#             if current_length >= MODEL_CONTEXT_LENGTH:
#                 break

#             next_input = torch.tensor(
#                 [[next_token]],
#                 dtype=torch.long,
#                 device=device
#             )

#             logits, past_key_values = model(
#                 next_input,
#                 past_key_values=past_key_values,
#                 use_cache=True
#             )

#             next_logits = logits[:, -1, :][0]

#     generated_text = tokenizer.decode(
#         generated_tokens
#     )

#     generated_words = generated_text.split()

#     return " ".join(
#         generated_words[:target_words]
#     )


# def generate_batch(
#     prompts,
#     model,
#     tokenizer,
#     device,
#     target_words=128
# ):
#     predictions = []

#     for prompt in prompts:
#         prediction = generate_one(
#             prompt=prompt,
#             model=model,
#             tokenizer=tokenizer,
#             device=device,
#             target_words=target_words
#         )

#         predictions.append(prediction)

#     return predictions


# print("Custom V1 generation functions ready.")
# print("Generation uses the KV cache.")

## 20. Generate Predictions for the Test Set

In [ ]:
# # Generate predictions for the complete test set.

# predictions = {}

# GENERATION_BATCH_SIZE = 32

# items = list(test_data.items())

# total_examples = len(items)

# print(
#     f"Total test examples: {total_examples}"
# )

# for start in range(
#     0,
#     total_examples,
#     GENERATION_BATCH_SIZE
# ):

#     batch = items[
#         start:start + GENERATION_BATCH_SIZE
#     ]

#     batch_ids = [
#         str(example_id)
#         for example_id, _ in batch
#     ]

#     batch_prompts = [
#         record["text"]
#         for _, record in batch
#     ]

#     batch_predictions = generate_batch(
#         prompts=batch_prompts,
#         model=generation_model,
#         tokenizer=generation_tokenizer,
#         device=DEVICE,
#         target_words=128
#     )

#     for example_id, prediction in zip(
#         batch_ids,
#         batch_predictions
#     ):

#         predictions[example_id] = {
#             "text": prediction
#         }

#     completed = start + len(batch)

#     print(
#         f"Completed {completed}/{total_examples} "
#         f"({100 * completed / total_examples:.1f}%)"
#     )

# print("\nTest generation complete.")
# print(
#     f"Generated predictions: {len(predictions)}"
# )

## 21. Save prediction.json

In [ ]:
# with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
#     json.dump(
#         predictions,
#         f,
#         ensure_ascii=False,
#         indent=2
#     )

# print("prediction.json saved:")
# print(PREDICTION_PATH)

## 22. Create CodaBench Submission ZIP

In [ ]:
# with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
#     zipf.write(PREDICTION_PATH, arcname="prediction.json")

# print("Codabench submission created:")
# print(SUBMISSION_ZIP)

## Development status

Implemented so far: resilient HF/local Phase 2 file resolution, V1 checkpoint restoration, labelled-data loading and validation, document-level train/validation split, and exact-token-ID character-offset verification. No Phase 2 prediction head or training loop has been added yet. The next step depends on the observed span-boundary alignment audit. Keep the course's individual-implementation requirements in mind when building the model architecture.

## Phase 2: EXP-1 V1 Full Fine-Tuning Baseline Overview

This extension integrates the pretrained Phase 1 V1 Transformer into an end-to-end Legal Named Entity Recognition (L-NER) pipeline for **EXP-1: Full Fine-Tuning**, while strictly preserving all original 85 cells of the baseline notebook.

### Key Pipeline Components:
1. **Data Diagnostics & Boundary Audit (Cell 86)**: Validates annotation dictionaries, sub-token alignment, and sliding-window boundary coverage.
2. **Contextual Hidden-State Extraction (Cell 87)**: Taps the 12-layer V1 Transformer backbone to extract $N_{\text{embd}}=768$ representations.
3. **Academic Compliance & Student Architecture Point (Cells 88–89)**: Identifies the modular task-specific prediction head (`StudentLinearNERHead`) marked with `# TODO [STUDENT IMPLEMENTATION]`.
4. **Offset-Aware Dataset & Collation (Cell 90)**: Window dataset with dynamic batch padding and non-destructive checkpointing.
5. **Training Engine & Validation (Cell 91)**: Epoch training loop, gradient clipping, finite loss validation, and strict span evaluation.
6. **Strict-Span Evaluation Suite (Cell 92)**: Verification of exact vs. off-by-one span matching and per-class performance metrics.
7. **Test Inference & CodaBench Packaging (Cell 93)**: Full inference over Fold-3 test documents and generation of `submission.zip`.
8. **Pre-Flight Smoke Test Suite (Cell 94)**: Automated sanity checks verifying loss finiteness, checkpoint isolation, and submission validity.

In [ ]:
# ==============================================================================
# Phase 2 Data Diagnostics Suite: Alignment, Offsets, and Windowing Integrity
# Dependencies:
#   - Cell 15: tokenizer (ByteLevelBPE25k instance)
#   - Cell 34: TEXTS_BY_ID, ANNOTATIONS_BY_ID, TRAIN_DOC_IDS, VAL_DOC_IDS, LABELS
#   - Cell 36: encode_with_offsets(text, tokenizer, ...)
#   - Cell 40: prepare_lner_windows(document_id, text, entities, max_length, ...)
# ==============================================================================

import re
from collections import Counter


def audit_annotation_boundary_integrity(texts_by_id, annotations_by_id, expected_labels=None):
    """
    Audit gold annotations against raw document strings:
    - Supports both dictionary annotations and tuple representations
    - Identifies offsets outside document bounds or inverted (start >= end)
    - Flags unrecognized entity labels
    - Compares annotated surface text (if present) against raw document slices
    """
    valid_labels = set(expected_labels if expected_labels is not None else LABELS)
    issues = {
        "total_documents_checked": len(annotations_by_id),
        "total_entities_checked": 0,
        "valid_entities": 0,
        "out_of_bounds": [],
        "inverted_or_empty": [],
        "unknown_labels": [],
        "surface_text_mismatches": [],
    }

    for doc_id, entities in annotations_by_id.items():
        doc_text = texts_by_id.get(str(doc_id), "")
        doc_len = len(doc_text)

        for idx, ent in enumerate(entities):
            issues["total_entities_checked"] += 1
            if isinstance(ent, dict):
                start = ent.get("start", ent.get("start_char", ent.get("start_offset", ent.get("begin"))))
                end = ent.get("end", ent.get("end_char", ent.get("end_offset", ent.get("stop"))))
                label = ent.get("label", ent.get("entity_label", ent.get("type", ent.get("entity_type"))))
                surface_gold = ent.get("text", ent.get("entity_text", ent.get("surface")))
            elif isinstance(ent, (list, tuple)) and len(ent) >= 3:
                start, end, label = ent[0], ent[1], ent[2]
                surface_gold = ent[3] if len(ent) > 3 else None
            else:
                continue

            if start < 0 or end > doc_len:
                issues["out_of_bounds"].append((doc_id, idx, start, end, doc_len))
                continue

            if start >= end:
                issues["inverted_or_empty"].append((doc_id, idx, start, end))
                continue

            if label not in valid_labels:
                issues["unknown_labels"].append((doc_id, idx, label))
                continue

            if surface_gold is not None:
                actual_slice = doc_text[start:end]
                if actual_slice != surface_gold:
                    issues["surface_text_mismatches"].append({
                        "doc_id": doc_id,
                        "entity_idx": idx,
                        "start": start,
                        "end": end,
                        "expected": surface_gold,
                        "actual": actual_slice,
                    })

            issues["valid_entities"] += 1

    return issues


def audit_tokenizer_span_alignment(texts_by_id, annotations_by_id, tokenizer_obj, sample_limit=50):
    """
    Evaluates how accurately ByteLevelBPE token boundaries align with gold character spans.
    Provides diagnostics for start-boundary, end-boundary, and unaligned entities.
    """
    total_spans = 0
    start_aligned = 0
    end_aligned = 0
    both_aligned = 0
    unaligned_examples = []
    label_misalignment_counter = Counter()

    doc_ids = list(annotations_by_id.keys())[:sample_limit]

    for doc_id in doc_ids:
        doc_text = texts_by_id.get(str(doc_id), "")
        if not doc_text:
            continue

        token_ids, offsets = encode_with_offsets(doc_text, tokenizer_obj, max_len=None) if 'max_len' in encode_with_offsets.__code__.co_varnames else encode_with_offsets(doc_text, tokenizer_obj)
        valid_offsets = [
            span for span in offsets
            if span is not None
            and isinstance(span, (tuple, list))
            and len(span) >= 2
            and span[0] is not None
            and span[1] is not None
        ]
        token_starts = {s for s, e in valid_offsets}
        token_ends = {e for s, e in valid_offsets}

        for ent in annotations_by_id[doc_id]:
            total_spans += 1
            if isinstance(ent, dict):
                s, e, lbl = ent["start"], ent["end"], ent["label"]
            else:
                s, e, lbl = ent[0], ent[1], ent[2]

            s_ok = s in token_starts
            e_ok = e in token_ends

            if s_ok:
                start_aligned += 1
            if e_ok:
                end_aligned += 1
            if s_ok and e_ok:
                both_aligned += 1
            else:
                label_misalignment_counter[lbl] += 1
                if len(unaligned_examples) < 10:
                    unaligned_examples.append({
                        "doc_id": doc_id,
                        "label": lbl,
                        "span": (s, e),
                        "text": doc_text[s:e],
                        "start_aligned": s_ok,
                        "end_aligned": e_ok,
                    })

    both_rate = (both_aligned / max(1, total_spans)) * 100
    start_rate = (start_aligned / max(1, total_spans)) * 100
    end_rate = (end_aligned / max(1, total_spans)) * 100

    return {
        "total_sampled_spans": total_spans,
        "start_aligned_count": start_aligned,
        "start_aligned_rate": start_rate,
        "end_aligned_count": end_aligned,
        "end_aligned_rate": end_rate,
        "both_aligned_count": both_aligned,
        "both_aligned_rate": both_rate,
        "unaligned_by_label": dict(label_misalignment_counter),
        "sample_unaligned": unaligned_examples,
    }


def audit_windowing_boundary_integrity(texts_by_id, annotations_by_id, max_len=512, window_overlap=256, sample_limit=30):
    """
    Examines long-document chunking across sliding windows using Cell 40 prepare_lner_windows:
    - Verifies document IDs are preserved
    - Evaluates entity coverage across window splits
    """
    total_docs = 0
    total_windows = 0
    total_entities = 0
    captured_entities = 0
    boundary_crossings = 0

    doc_ids = list(annotations_by_id.keys())[:sample_limit]

    for doc_id in doc_ids:
        text = texts_by_id.get(str(doc_id), "")
        entities = annotations_by_id[doc_id]
        windows, report = prepare_lner_windows(
            document_id=doc_id,
            text=text,
            entities=entities,
            max_length=max_len,
            window_overlap=window_overlap,
            tokenizer=tokenizer
        )
        total_docs += 1
        total_windows += len(windows)
        total_entities += len(entities)

        covered_entities = set()
        for win in windows:
            valid_offsets = [
                (span[0], span[1])
                for span in win["offsets"]
                if span is not None
                and isinstance(span, (tuple, list))
                and len(span) >= 2
                and span[0] is not None
                and span[1] is not None
            ]
            if not valid_offsets:
                continue
            win_char_start = min(st for st, en in valid_offsets)
            win_char_end = max(en for st, en in valid_offsets)

            for ent_idx, ent in enumerate(entities):
                s, e, lbl = (ent["start"], ent["end"], ent["label"]) if isinstance(ent, dict) else (ent[0], ent[1], ent[2])
                if s >= win_char_start and e <= win_char_end:
                    covered_entities.add(ent_idx)
                elif (s < win_char_end and e > win_char_start) and not (s >= win_char_start and e <= win_char_end):
                    boundary_crossings += 1

        captured_entities += len(covered_entities)

    return {
        "documents_audited": total_docs,
        "total_windows_generated": total_windows,
        "gold_entities_count": total_entities,
        "fully_covered_entities": captured_entities,
        "boundary_crossings_observed": boundary_crossings,
    }


def run_full_phase2_data_diagnostics():
    print("=" * 70)
    print("PHASE 2 DATA DIAGNOSTICS & ALIGNMENT AUDIT")
    print("=" * 70)

    annot_issues = audit_annotation_boundary_integrity(TEXTS_BY_ID, ANNOTATIONS_BY_ID)
    print(f"[1] Annotation Integrity:")
    print(f"    Total Checked: {annot_issues['total_entities_checked']} across {annot_issues['total_documents_checked']} docs")
    print(f"    Valid Entities: {annot_issues['valid_entities']}")
    print(f"    Out of bounds: {len(annot_issues['out_of_bounds'])}")
    print(f"    Inverted/empty: {len(annot_issues['inverted_or_empty'])}")
    print(f"    Unknown labels: {len(annot_issues['unknown_labels'])}")
    print(f"    Surface text mismatches: {len(annot_issues['surface_text_mismatches'])}")

    align_stats = audit_tokenizer_span_alignment(TEXTS_BY_ID, ANNOTATIONS_BY_ID, tokenizer, sample_limit=50)
    print(f"[2] Tokenizer Span Alignment (50 sample docs):")
    print(f"    Total Spans Audited: {align_stats['total_sampled_spans']}")
    print(f"    Start-Boundary Aligned: {align_stats['start_aligned_count']} ({align_stats['start_aligned_rate']:.2f}%)")
    print(f"    End-Boundary Aligned:   {align_stats['end_aligned_count']} ({align_stats['end_aligned_rate']:.2f}%)")
    print(f"    Both Boundaries Aligned: {align_stats['both_aligned_count']} ({align_stats['both_aligned_rate']:.2f}%)")

    win_stats = audit_windowing_boundary_integrity(TEXTS_BY_ID, ANNOTATIONS_BY_ID, max_len=MODEL_CONTEXT_LENGTH, window_overlap=MODEL_CONTEXT_LENGTH // 2, sample_limit=30)
    print(f"[3] Windowing Integrity (30 sample docs):")
    print(f"    Docs Audited: {win_stats['documents_audited']}")
    print(f"    Windows Generated: {win_stats['total_windows_generated']}")
    print(f"    Fully Captured Entities: {win_stats['fully_covered_entities']} / {win_stats['gold_entities_count']}")
    print(f"    Boundary Crossings (Resolved via window overlap): {win_stats['boundary_crossings_observed']}")
    print("=" * 70)
    return annot_issues, align_stats, win_stats

diagnostics_results = run_full_phase2_data_diagnostics()

In [ ]:
# ==============================================================================
# Contextual Hidden-State Extraction from Trained V1 Model
# Dependencies:
#   - Cell 9: N_EMBD (768)
#   - Cell 29: V1LegalLM, RMSNorm, TransformerBlock
#   - Cell 30: DEVICE
#   - Cell 31: model (V1LegalLM instance restored from v1_model.pt)
# ==============================================================================

import torch
import torch.nn as nn


def extract_v1_hidden_states(v1_model, input_ids, attention_mask=None):
    """
    Extracts contextual representation vectors from the pretrained V1 backbone.
    Preserves token representations across all 12 Transformer blocks up to final RMSNorm.
    
    Args:
        v1_model (V1LegalLM): Pretrained Phase 1 model instance
        input_ids (torch.Tensor): LongTensor of shape (batch_size, seq_len)
        attention_mask (torch.Tensor, optional): Float/BoolTensor of shape (batch_size, seq_len)
        
    Returns:
        torch.Tensor: Contextual hidden states of shape (batch_size, seq_len, N_EMBD)
    """
    B, T = input_ids.shape

    # 1. Token Embeddings
    x = v1_model.token_embedding(input_ids)  # Shape: (B, T, N_EMBD)

    # 2. Sequential Transformer Blocks
    for block in v1_model.blocks:
        x, _ = block(x, past_key_value=None, use_cache=False)

    # 3. Final Pre-LM Normalization Layer
    hidden_states = v1_model.final_norm(x)  # Shape: (B, T, N_EMBD)

    return hidden_states


def verify_v1_hidden_extraction():
    model.eval()
    dummy_input = torch.randint(0, tokenizer.vocab_size, (2, 32), device=DEVICE)
    with torch.no_grad():
        h = extract_v1_hidden_states(model, dummy_input)
    assert h.shape == (2, 32, N_EMBD), f"Unexpected shape {h.shape}, expected (2, 32, {N_EMBD})"
    print(f"[OK] extract_v1_hidden_states verified. Output shape: {h.shape}")

verify_v1_hidden_extraction()

## Phase 2 Architecture Scaffold & Academic Compliance Notice


In [ ]:
# ==============================================================================
# Phase 2 Model Scaffold, Base Classes, and Integration Interfaces
# ==============================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------------------------
# 1. Base Prediction Head Interface
# ------------------------------------------------------------------------------
class BasePhase2NERHead(nn.Module):
    def forward(self, hidden_states, attention_mask=None):
        raise NotImplementedError(
            "Implement forward() in your custom prediction head subclass."
        )


# ------------------------------------------------------------------------------
# 2. Student Linear Projection Head
# ------------------------------------------------------------------------------
class StudentLinearNERHead(BasePhase2NERHead):
    """
    Token-level BIO classification head.

    Maps contextual backbone hidden states to BIO label logits.
    """
    def __init__(
        self,
        hidden_dim=N_EMBD,
        num_classes=len(BIO_LABELS),
        dropout_prob=0.1,
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, hidden_states, attention_mask=None):
        hidden_states = self.dropout(hidden_states)
        logits = self.classifier(hidden_states)

        return logits


# ------------------------------------------------------------------------------
# 3. Integration Container: Backbone + NER Head
# ------------------------------------------------------------------------------
class V1Phase2NERModel(nn.Module):
    """
    Connects the Phase 1 V1 backbone to a Phase 2 NER prediction head.
    """
    def __init__(self, backbone, ner_head, freeze_backbone=False):
        super().__init__()

        self.backbone = backbone
        self.ner_head = ner_head
        self.freeze_backbone = freeze_backbone

        if freeze_backbone:
            for parameter in self.backbone.parameters():
                parameter.requires_grad = False

    def forward(self, input_ids, attention_mask=None):
        if self.freeze_backbone:
            with torch.no_grad():
                hidden_states = extract_v1_hidden_states(
                    self.backbone,
                    input_ids,
                    attention_mask,
                )
        else:
            hidden_states = extract_v1_hidden_states(
                self.backbone,
                input_ids,
                attention_mask,
            )

        logits = self.ner_head(hidden_states, attention_mask)

        return logits


# ------------------------------------------------------------------------------
# 4. BIO Decoding with Safe Offset Handling
# ------------------------------------------------------------------------------
def decode_token_bio_predictions(
    logits,
    window_offsets,
    raw_text=None,
    id_to_bio=ID_TO_BIO,
    refine_boundaries=True,
):
    """
    Convert token-level BIO logits into character-level spans.

    Args:
        logits: Tensor [batch_size, sequence_length, num_labels].
        window_offsets: Per-window token offsets into the original document.
        raw_text: Original document text, if available.
        id_to_bio: Mapping from class IDs to BIO labels.
        refine_boundaries: Trim surrounding whitespace and punctuation.

    Returns:
        List of (start, end, label) tuples.

    Note:
        This decoder cannot recover arbitrary character boundaries inside
        tokenizer tokens. Boundary trimming is not a substitute for a
        character-boundary prediction model.
    """
    predictions = (
        torch.argmax(logits, dim=-1)
        .detach()
        .cpu()
        .tolist()
    )

    extracted_spans = []

    for batch_idx, sequence_predictions in enumerate(predictions):
        if batch_idx >= len(window_offsets):
            break

        offsets = window_offsets[batch_idx]

        current_label = None
        current_start = None
        current_end = None

        def flush_current():
            nonlocal current_label, current_start, current_end

            if (
                current_label is not None
                and current_start is not None
                and current_end is not None
                and current_start < current_end
            ):
                extracted_spans.append(
                    (current_start, current_end, current_label)
                )

            current_label = None
            current_start = None
            current_end = None

        for token_idx, label_id in enumerate(sequence_predictions):
            if token_idx >= len(offsets):
                flush_current()
                break

            span = offsets[token_idx]

            if (
                span is None
                or not isinstance(span, (tuple, list))
                or len(span) < 2
                or span[0] is None
                or span[1] is None
            ):
                flush_current()
                continue

            try:
                token_start = int(span[0])
                token_end = int(span[1])
                label_id = int(label_id)
            except (TypeError, ValueError):
                flush_current()
                continue

            if token_start < 0 or token_start >= token_end:
                flush_current()
                continue

            bio_tag = id_to_bio.get(label_id, "O")

            if bio_tag.startswith("B-"):
                flush_current()

                current_label = bio_tag[2:]
                current_start = token_start
                current_end = token_end

            elif bio_tag.startswith("I-"):
                entity_type = bio_tag[2:]

                if current_label == entity_type:
                    # Preserve normal whitespace gaps between token offsets.
                    current_end = max(current_end, token_end)
                else:
                    flush_current()

                    # Treat an I-tag without a matching preceding B-tag
                    # as the beginning of a new entity.
                    current_label = entity_type
                    current_start = token_start
                    current_end = token_end

            else:
                flush_current()

        flush_current()

    # ------------------------------------------------------------------
    # Optional boundary cleanup
    # ------------------------------------------------------------------
    if raw_text is None or not refine_boundaries:
        return extracted_spans

    refined_spans = []
    strip_chars = " \t\n\r.,;:()[]{}'\""

    for start, end, label in extracted_spans:
        start = max(0, int(start))
        end = min(len(raw_text), int(end))

        if start >= end:
            continue

        span_text = raw_text[start:end]
        trimmed_text = span_text.strip(strip_chars)

        if not trimmed_text:
            continue

        leading_whitespace = span_text.find(trimmed_text)
        refined_start = start + leading_whitespace
        refined_end = refined_start + len(trimmed_text)

        if refined_start < refined_end:
            refined_spans.append(
                (refined_start, refined_end, label)
            )

    return refined_spans


print("Phase 2 model scaffold and decoder ready.")
print("Classes: BasePhase2NERHead, StudentLinearNERHead, V1Phase2NERModel.")
print("Decoder: decode_token_bio_predictions.")

In [ ]:
# ==============================================================================
# Phase 2 Training Infrastructure: Optimizer, Schedulers, and Checkpointing
# Dependencies:
#   - Cell 6: SEED
#   - Cell 10: LEARNING_RATE, WEIGHT_DECAY
#   - Cell 12: EXPERIMENT_NAME, PHASE2_ROOT_DIR, PHASE2_CHECKPOINT_DIR
#   - Cell 34: TRAIN_DOC_IDS, VAL_DOC_IDS, TEXTS_BY_ID, ANNOTATIONS_BY_ID
#   - Cell 40: prepare_lner_windows, IGNORE_INDEX
# ==============================================================================

import os
import torch
from torch.utils.data import Dataset, DataLoader


class Phase2NERDataset(Dataset):
    """
    PyTorch Dataset for Phase 2 L-NER windows.
    Stores input_ids, labels, document_id, token_start, and character offsets
    for exact span decoding and evaluation.
    """
    def __init__(self, document_ids, text_by_id, entities_by_id, tokenizer_obj=None,
                 max_length=512, window_overlap=256):
        self.samples = []
        active_tok = tokenizer_obj if tokenizer_obj is not None else globals().get("tokenizer")

        for doc_id in document_ids:
            doc_str = str(doc_id)
            text = text_by_id.get(doc_str, "")
            if not text:
                continue
            entities = entities_by_id.get(doc_str, [])
            windows, report = prepare_lner_windows(
                document_id=doc_str,
                text=text,
                entities=entities,
                max_length=max_length,
                window_overlap=window_overlap,
                tokenizer=active_tok,
            )
            for w in windows:
                self.samples.append({
                    "input_ids": torch.tensor(w["input_ids"], dtype=torch.long),
                    "labels": torch.tensor(w["labels"], dtype=torch.long),
                    "offsets": w["offsets"],
                    "document_id": w["document_id"],
                    "token_start": w["token_start"],
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_phase2_batch(batch, pad_token_id=0, ignore_index=IGNORE_INDEX):
    """
    Pads dynamic sequences within batch and constructs binary attention masks.
    """
    max_len = max(item["input_ids"].size(0) for item in batch)
    B = len(batch)

    padded_input_ids = torch.full((B, max_len), pad_token_id, dtype=torch.long)
    padded_labels = torch.full((B, max_len), ignore_index, dtype=torch.long)
    attention_mask = torch.zeros((B, max_len), dtype=torch.float32)

    document_ids = []
    offsets = []
    token_starts = []

    for i, item in enumerate(batch):
        seq_len = item["input_ids"].size(0)
        padded_input_ids[i, :seq_len] = item["input_ids"]
        padded_labels[i, :seq_len] = item["labels"]
        attention_mask[i, :seq_len] = 1.0

        document_ids.append(item["document_id"])
        offsets.append(item["offsets"])
        token_starts.append(item["token_start"])

    return {
        "input_ids": padded_input_ids,
        "labels": padded_labels,
        "attention_mask": attention_mask,
        "document_ids": document_ids,
        "offsets": offsets,
        "token_starts": token_starts,
    }


def create_phase2_optimizer(model, lr=3e-5, weight_decay=0.01):
    """
    Creates AdamW optimizer with decoupled weight decay for 1D parameters (biases, norms).
    """
    decay_params = []
    no_decay_params = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2:
            decay_params.append(p)
        else:
            no_decay_params.append(p)

    groups = []
    if decay_params:
        groups.append({"params": decay_params, "weight_decay": weight_decay, "lr": lr})
    if no_decay_params:
        groups.append({"params": no_decay_params, "weight_decay": 0.0, "lr": lr})

    return torch.optim.AdamW(groups)


def save_phase2_checkpoint(model, optimizer, epoch, val_f1, save_path):
    """
    Saves isolated Phase 2 training checkpoint without overwriting Phase 1 weights.
    """
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    state = {
        "epoch": epoch,
        "val_f1": val_f1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer else None,
    }
    torch.save(state, save_path)
    print(f"[Checkpoint] Saved Phase 2 checkpoint to {save_path} (Val F1: {val_f1:.4f})")


def restore_phase2_checkpoint(checkpoint_path, model, optimizer=None):
    """
    Restores Phase 2 checkpoint with strict state verification.
    """
    if not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    state = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(state["model_state_dict"], strict=True)
    if optimizer and state.get("optimizer_state_dict"):
        optimizer.load_state_dict(state["optimizer_state_dict"])
    print(f"[Checkpoint] Restored Phase 2 checkpoint from {checkpoint_path} (Epoch: {state.get('epoch')}, Val F1: {state.get('val_f1', 0.0):.4f})")
    return state

print("Phase 2 Training Infrastructure ready.")
print("Utilities: Phase2NERDataset, collate_phase2_batch, create_phase2_optimizer, save_phase2_checkpoint, restore_phase2_checkpoint.")

In [ ]:
# ==============================================================================
# Phase 2 Training Engine and Validation Loop
# Dependencies:
#   - DEVICE
#   - strict_span_metrics(gold_by_id, predicted_by_id, ...)
#   - decode_token_bio_predictions
#   - collate_phase2_batch
#   - create_phase2_optimizer
#   - save_phase2_checkpoint
# ==============================================================================

import time
import torch
import torch.nn.functional as F


def train_one_epoch(
    model,
    dataloader,
    optimizer,
    criterion,
    device=DEVICE,
    clip_norm=1.0,
):
    """Run one training epoch."""

    model.train()
    total_loss = 0.0
    num_batches = 0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(
            input_ids,
            attention_mask=attention_mask,
        )

        B, T, num_classes = logits.shape

        loss = criterion(
            logits.reshape(-1, num_classes),
            labels.reshape(-1),
        )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite loss encountered: "
                f"{loss.item()} at batch {num_batches}"
            )

        loss.backward()

        if clip_norm > 0:
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=clip_norm,
            )

        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    if num_batches == 0:
        raise RuntimeError(
            "Training DataLoader produced no batches."
        )

    return total_loss / num_batches


def evaluate_phase2_model(
    model,
    val_dataset,
    texts_by_id,
    gold_by_id,
    device=DEVICE,
    batch_size=8,
    labels=LABELS,
):
    """
    Evaluate the model on held-out documents.

    Returns strict-span metrics and validation cross-entropy.
    """

    model.eval()

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_phase2_batch,
    )

    pred_spans_by_doc = {}

    # Accumulate validation loss over valid target tokens.
    val_loss_sum = 0.0
    val_token_count = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            doc_ids = batch["document_ids"]
            offsets = batch["offsets"]
            labels_batch = batch["labels"].to(device)

            logits = model(
                input_ids,
                attention_mask=attention_mask,
            )

            # Validation loss: ignore padding and ignored targets.
            loss_sum = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                labels_batch.reshape(-1),
                ignore_index=IGNORE_INDEX,
                reduction="sum",
            )

            valid_tokens = int(
                (labels_batch != IGNORE_INDEX).sum().item()
            )

            val_loss_sum += float(loss_sum.item())
            val_token_count += valid_tokens

            # Decode predictions back into character-level spans.
            for i in range(len(doc_ids)):
                doc_id = str(doc_ids[i])
                raw_text = texts_by_id.get(doc_id, "")

                single_logits = logits[i:i + 1]
                single_offsets = [offsets[i]]

                spans = decode_token_bio_predictions(
                    logits=single_logits,
                    window_offsets=single_offsets,
                    raw_text=raw_text,
                    id_to_bio=ID_TO_BIO,
                    refine_boundaries=True,
                )

                if doc_id not in pred_spans_by_doc:
                    pred_spans_by_doc[doc_id] = []

                pred_spans_by_doc[doc_id].extend(spans)

    if val_token_count == 0:
        raise RuntimeError(
            "Validation loss could not be calculated: "
            "no valid target tokens were found."
        )

    # Deduplicate identical predictions from overlapping windows.
    clean_preds_by_doc = {}

    for doc_id, spans in pred_spans_by_doc.items():
        seen = set()
        deduped = []

        for span in spans:
            key = (
                int(span[0]),
                int(span[1]),
                str(span[2]),
            )

            if key not in seen:
                seen.add(key)
                deduped.append(list(key))

        clean_preds_by_doc[doc_id] = deduped

    # Evaluate only documents represented in the prediction set.
    eval_gold_by_id = {
        str(doc_id): gold_by_id.get(str(doc_id), [])
        for doc_id in clean_preds_by_doc
    }

    raw_metrics = strict_span_metrics(
        gold_by_id=eval_gold_by_id,
        predicted_by_id=clean_preds_by_doc,
        labels=labels,
    )

    strict_f1 = raw_metrics.get("strict_macro_f1", 0.0)
    per_label = raw_metrics.get("per_label", {})

    precisions = [
        value.get("precision", 0.0)
        for value in per_label.values()
    ]

    recalls = [
        value.get("recall", 0.0)
        for value in per_label.values()
    ]

    macro_p = (
        sum(precisions) / len(precisions)
        if precisions else 0.0
    )

    macro_r = (
        sum(recalls) / len(recalls)
        if recalls else 0.0
    )

    return {
        "strict_macro_f1": strict_f1,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "val_loss": val_loss_sum / val_token_count,
        "per_label": per_label,
        "documents_evaluated": raw_metrics.get(
            "documents_evaluated",
            len(clean_preds_by_doc),
        ),
        "predictions_by_doc": clean_preds_by_doc,
    }


def run_phase2_training(
    model,
    train_dataset,
    val_dataset,
    texts_by_id,
    gold_by_id,
    epochs=3,
    lr=3e-5,
    batch_size=8,
    checkpoint_path=None,
    device=DEVICE,
    class_weights=None,
):
    """Train, validate, log metrics, and save the best checkpoint.

    Pass class_weights=None for ordinary cross-entropy or a 1D tensor
    aligned with BIO_LABELS for weighted cross-entropy.
    """

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_phase2_batch,
    )

    optimizer = create_phase2_optimizer(
        model,
        lr=lr,
    )

    criterion = torch.nn.CrossEntropyLoss(
        weight=(
            class_weights.to(device=device, dtype=torch.float32)
            if class_weights is not None else None
        ),
        ignore_index=IGNORE_INDEX,
    )

    best_f1 = -1.0
    history = []

    print(
        f"Starting Phase 2 Training "
        f"({epochs} epochs, lr={lr}, device={device})"
    )

    for ep in range(1, epochs + 1):
        t0 = time.time()

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device=device,
        )

        val_res = evaluate_phase2_model(
            model,
            val_dataset,
            texts_by_id,
            gold_by_id,
            device=device,
        )

        elapsed = time.time() - t0

        val_loss = val_res["val_loss"]
        val_f1 = val_res["strict_macro_f1"]
        val_p = val_res["macro_precision"]
        val_r = val_res["macro_recall"]

        history.append({
            "epoch": ep,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_f1": val_f1,
            "val_precision": val_p,
            "val_recall": val_r,
            "time_sec": elapsed,
        })

        print(
            f"Epoch {ep:02d}/{epochs:02d} "
            f"[{elapsed:.1f}s] - "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Strict Macro-F1: {val_f1:.4f} | "
            f"Precision: {val_p:.4f} | "
            f"Recall: {val_r:.4f}"
        )
        print("  Per-label validation F1:")
        for label_name, label_metrics in val_res.get("per_label", {}).items():
            print(
                f"    {label_name:12s} "
                f"F1={label_metrics.get('f1', 0.0):.3f} "
                f"P={label_metrics.get('precision', 0.0):.3f} "
                f"R={label_metrics.get('recall', 0.0):.3f} "
                f"(TP={label_metrics.get('tp', 0)}, "
                f"FP={label_metrics.get('fp', 0)}, "
                f"FN={label_metrics.get('fn', 0)})"
            )

        if checkpoint_path and val_f1 > best_f1:
            best_f1 = val_f1

            save_phase2_checkpoint(
                model,
                optimizer,
                ep,
                val_f1,
                checkpoint_path,
            )

    return history, best_f1


print("Phase 2 Training Engine and Validation Loop ready.")
print(
    "Functions: train_one_epoch, "
    "evaluate_phase2_model, run_phase2_training."
)

In [ ]:
# ==============================================================================
# Strict Span Evaluation Verification Suite & Comprehensive Metrics
# Dependencies:
#   - Cell 34: LABELS
#   - Cell 46: strict_span_metrics(gold_by_id, predicted_by_id, ...)
# ==============================================================================


def verify_strict_evaluator_properties():
    """
    Verifies properties of strict span evaluation:
    1. Exact match returns strict_macro_f1 = 1.0 on evaluated classes.
    2. Off-by-one character shifts produce strict_macro_f1 = 0.0 under exact matching.
    3. Label mismatches produce strict_macro_f1 = 0.0.
    """
    test_labels = ["COURT", "JUDGE"]

    gold = {
        "doc_0": [{"start": 0, "end": 10, "label": "COURT"}, {"start": 20, "end": 30, "label": "JUDGE"}]
    }

    # Case 1: Exact Match
    exact_pred = {
        "doc_0": [(0, 10, "COURT"), (20, 30, "JUDGE")]
    }
    m_exact = strict_span_metrics(gold, exact_pred, labels=test_labels)
    assert m_exact["strict_macro_f1"] == 1.0, f"Expected 1.0, got {m_exact['strict_macro_f1']}"
    assert m_exact["per_label"]["COURT"]["f1"] == 1.0
    assert m_exact["per_label"]["JUDGE"]["f1"] == 1.0

    # Case 2: Off-by-one character shift (strict match failure)
    shifted_pred = {
        "doc_0": [(0, 11, "COURT"), (20, 29, "JUDGE")]
    }
    m_shift = strict_span_metrics(gold, shifted_pred, labels=test_labels)
    assert m_shift["strict_macro_f1"] == 0.0, f"Expected 0.0 for shifted spans, got {m_shift['strict_macro_f1']}"
    assert m_shift["per_label"]["COURT"]["tp"] == 0

    # Case 3: Label mismatch
    wrong_label_pred = {
        "doc_0": [(0, 10, "JUDGE"), (20, 30, "COURT")]
    }
    m_label = strict_span_metrics(gold, wrong_label_pred, labels=test_labels)
    assert m_label["strict_macro_f1"] == 0.0, f"Expected 0.0 for label mismatches, got {m_label['strict_macro_f1']}"

    print("[OK] Strict Span Evaluator Properties successfully verified on synthetic test cases.")
    return True


def compute_comprehensive_strict_metrics(gold_by_id, predicted_by_id, label_list=None):
    """
    Computes comprehensive per-class and aggregated strict span metrics:
    - strict_macro_f1 (official capstone metric across all classes)
    - strict_macro_precision
    - strict_macro_recall
    - active_macro_f1 (macro average restricted to classes present in gold or pred)
    - aggregate true positives, false positives, false negatives
    """
    target_labels = label_list if label_list is not None else LABELS
    base_metrics = strict_span_metrics(gold_by_id, predicted_by_id, labels=target_labels)

    per_label = base_metrics.get("per_label", {})
    active_f1s = []
    precisions = []
    recalls = []
    total_tp = 0
    total_fp = 0
    total_fn = 0

    for lbl, scores in per_label.items():
        tp = scores.get("tp", 0)
        fp = scores.get("fp", 0)
        fn = scores.get("fn", 0)
        p = scores.get("precision", 0.0)
        r = scores.get("recall", 0.0)
        total_tp += tp
        total_fp += fp
        total_fn += fn
        precisions.append(p)
        recalls.append(r)

        if (tp + fp + fn) > 0:
            active_f1s.append(scores.get("f1", 0.0))

    strict_macro_f1 = base_metrics.get("strict_macro_f1", 0.0)
    macro_p = (sum(precisions) / len(precisions)) if precisions else 0.0
    macro_r = (sum(recalls) / len(recalls)) if recalls else 0.0
    active_macro_f1 = (sum(active_f1s) / len(active_f1s)) if active_f1s else 0.0

    return {
        "strict_macro_f1": strict_macro_f1,
        "strict_macro_precision": macro_p,
        "strict_macro_recall": macro_r,
        "active_macro_f1": active_macro_f1,
        "total_true_positives": total_tp,
        "total_false_positives": total_fp,
        "total_false_negatives": total_fn,
        "per_label": per_label,
    }

verify_strict_evaluator_properties()

In [ ]:
# ==============================================================================
# End-to-End Test Inference & CodaBench Submission Packaging
# Dependencies:
#   - Cell 12: PHASE2_OUTPUT_DIR, PHASE2_PREDICTION_PATH, PHASE2_SUBMISSION_ZIP
#   - Cell 15: tokenizer
#   - Cell 34: TEST_TEXTS, LABELS
#   - Cell 40: prepare_lner_windows
#   - Cell 46: write_lner_submission
#   - Cell 89: decode_token_bio_predictions
# ==============================================================================

import os
import json
import zipfile
import torch


def run_phase2_test_inference(model, test_texts_by_id, device=DEVICE, max_length=512,
                              window_overlap=256, batch_size=8, tokenizer_obj=None):
    """
    Executes inference over test documents (Fold 3):
    - Preserves EVERY required test document ID in test_texts_by_id (even if 0 spans found -> [])
    - Uses exact full-document character offsets
    - Recombines sliding windows and deduplicates span predictions
    - Emits standard JSON structure: {"doc_id": [[start, end, label], ...]}
    """
    model.eval()
    active_tok = tokenizer_obj if tokenizer_obj is not None else globals().get("tokenizer")
    all_predictions = {}
    test_doc_ids = sorted(map(str, test_texts_by_id.keys()))

    print(f"Running inference across {len(test_doc_ids)} test documents...")

    for doc_id in test_doc_ids:
        raw_text = test_texts_by_id[doc_id]
        if not raw_text.strip():
            all_predictions[doc_id] = []
            continue

        windows, _ = prepare_lner_windows(
            document_id=doc_id,
            text=raw_text,
            entities=[],
            max_length=max_length,
            window_overlap=window_overlap,
            tokenizer=active_tok,
        )

        if not windows:
            all_predictions[doc_id] = []
            continue

        doc_spans = []
        for i in range(0, len(windows), batch_size):
            chunk = windows[i:i + batch_size]
            B = len(chunk)
            max_chunk_len = max(len(w["input_ids"]) for w in chunk)

            input_ids = torch.zeros((B, max_chunk_len), dtype=torch.long)
            attn_mask = torch.zeros((B, max_chunk_len), dtype=torch.float32)

            for b_idx, w in enumerate(chunk):
                l = len(w["input_ids"])
                input_ids[b_idx, :l] = torch.tensor(w["input_ids"], dtype=torch.long)
                attn_mask[b_idx, :l] = 1.0

            input_ids = input_ids.to(device)
            attn_mask = attn_mask.to(device)

            with torch.no_grad():
                logits = model(input_ids, attention_mask=attn_mask)

            for b_idx, w in enumerate(chunk):
                single_logits = logits[b_idx:b_idx + 1]
                single_offsets = [w["offsets"]]
                spans = decode_token_bio_predictions(
                    logits=single_logits,
                    window_offsets=single_offsets,
                    raw_text=raw_text,
                    id_to_bio=ID_TO_BIO,
                    refine_boundaries=True
                )
                doc_spans.extend(spans)

        # Deduplicate predictions across windows
        seen = set()
        clean_spans = []
        for s in doc_spans:
            key = (int(s[0]), int(s[1]), str(s[2]))
            if key not in seen:
                seen.add(key)
                clean_spans.append([int(s[0]), int(s[1]), str(s[2])])

        clean_spans.sort(key=lambda x: (x[0], x[1]))
        all_predictions[doc_id] = clean_spans

    return all_predictions


def package_codabench_submission(predictions, output_dir=PHASE2_OUTPUT_DIR, zip_filename="submission.zip"):
    """
    Packages predictions strictly per CodaBench format:
    submission.zip
    └── prediction.json (directly at root of zip, no enclosing folders)
    """
    os.makedirs(output_dir, exist_ok=True)
    json_path = os.path.join(output_dir, "prediction.json")
    zip_path = os.path.join(output_dir, zip_filename)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=2, ensure_ascii=False)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(json_path, arcname="prediction.json")

    print(f"[Submission] Saved prediction.json to {json_path}")
    print(f"[Submission] Created CodaBench package at {zip_path}")
    return json_path, zip_path


def validate_submission_zip(zip_path, expected_doc_ids=None):
    """
    Validates submission ZIP integrity against competition requirements:
    - prediction.json exists directly at root of the archive
    - All expected document IDs are present
    - Span format is [start, end, label] with valid integers and labels
    """
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f"Submission zip does not exist: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as zf:
        namelist = zf.namelist()
        if "prediction.json" not in namelist:
            raise ValueError(f"prediction.json missing from root of {zip_path}. Found: {namelist}")

        content = zf.read("prediction.json").decode("utf-8")
        preds = json.loads(content)

    if not isinstance(preds, dict):
        raise TypeError("prediction.json must contain a top-level JSON dictionary of document IDs.")

    if expected_doc_ids is not None:
        expected_set = {str(d) for d in expected_doc_ids}
        found_set = set(preds.keys())
        missing = expected_set - found_set
        extra = found_set - expected_set
        if missing:
            raise ValueError(f"Submission missing {len(missing)} required document IDs. Examples: {list(missing)[:5]}")
        if extra:
            print(f"[Warning] Submission contains {len(extra)} unexpected document IDs.")

    total_spans = 0
    valid_labels = set(LABELS)

    for doc_id, span_list in preds.items():
        if not isinstance(span_list, list):
            raise TypeError(f"Document {doc_id} predictions must be a list of spans.")
        for item in span_list:
            total_spans += 1
            if len(item) < 3:
                raise ValueError(f"Span {item} in doc {doc_id} must have at least 3 elements [start, end, label].")
            start, end, label = item[0], item[1], item[2]
            if not isinstance(start, int) or not isinstance(end, int):
                raise TypeError(f"Offsets must be integers, got start={start}, end={end} in doc {doc_id}")
            if start >= end:
                raise ValueError(f"Inverted span found: start={start} >= end={end} in doc {doc_id}")
            if label not in valid_labels:
                raise ValueError(f"Unknown entity label '{label}' in doc {doc_id}")
            if doc_id in TEST_TEXTS and not (0 <= start < end <= len(TEST_TEXTS[doc_id])):
                raise ValueError(
                    f"Span {(start, end, label)} is outside document {doc_id} "
                    f"(text length={len(TEST_TEXTS[doc_id])})."
                )

    print(f"[Verification] Submission ZIP validated successfully.")
    print(f"               Documents: {len(preds)}, Total Spans: {total_spans}")
    return True

print("Test Inference and Submission Packaging utilities ready.")
print("Functions: run_phase2_test_inference, package_codabench_submission, validate_submission_zip.")

In [ ]:
# ==============================================================================
# EXP-1 Pre-Flight Verification & Comprehensive Smoke Tests
# Dependencies:
#   - Cells 86 to 93
# ==============================================================================

import os
import shutil
import tempfile
import torch
import torch.nn as nn
import copy


def run_exp1_preflight_smoke_tests():
    print("=" * 70)
    print("RUNNING EXP-1 PRE-FLIGHT VERIFICATION & SMOKE TESTS")
    print("=" * 70)

    # Snapshot the real restored Phase 1 backbone so the smoke tests
    # can explicitly verify it remains untouched.
    original_backbone_state = {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }

    # 1. Forward and Backward Smoke Test on an independent copy.
    # Never optimize the global Phase 1 model during a preflight test.
    print("[Check 1/6] Testing V1Phase2NERModel forward and backward pass...")
    test_backbone = copy.deepcopy(model)
    test_head = StudentLinearNERHead(hidden_dim=N_EMBD, num_classes=len(BIO_LABELS))
    test_model = V1Phase2NERModel(test_backbone, test_head, freeze_backbone=False).to(DEVICE)
    test_model.train()

    dummy_input = torch.randint(0, tokenizer.vocab_size, (2, 16), device=DEVICE)
    dummy_labels = torch.full((2, 16), IGNORE_INDEX, dtype=torch.long, device=DEVICE)
    dummy_labels[0, :8] = O_LABEL_ID
    dummy_labels[0, 2] = BIO_TO_ID["B-COURT"]
    dummy_labels[0, 3] = BIO_TO_ID["I-COURT"]

    opt = torch.optim.AdamW(test_model.parameters(), lr=1e-4)
    crit = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

    opt.zero_grad()
    logits = test_model(dummy_input)
    loss = crit(logits.view(-1, len(BIO_LABELS)), dummy_labels.view(-1))
    assert torch.isfinite(loss).item(), f"Loss is non-finite: {loss.item()}"
    loss.backward()
    opt.step()
    print(f"  --> Forward/backward pass passed. Finite loss: {loss.item():.4f}")

    # 2. BIO Decoder Safe None-Offset Handling
    print("[Check 2/6] Testing BIO decoder with None offsets...")
    dummy_offsets = [
        [None, (0, 3), (4, 11), None, (12, 17)] + [None] * 11,
        [(0, 5), None, (6, 12)] + [None] * 13
    ]
    spans = decode_token_bio_predictions(logits, dummy_offsets, raw_text="The Supreme Court of India")
    print(f"  --> Decoded spans safely without exception: {len(spans)} spans extracted.")

    # 3. Strict Evaluator Verification
    print("[Check 3/6] Testing strict span evaluator...")
    verify_strict_evaluator_properties()
    print("  --> Strict evaluator verified.")

    # 4. Checkpoint Non-Destructive Isolation Test
    print("[Check 4/6] Testing non-destructive Phase 2 checkpointing...")
    temp_dir = tempfile.mkdtemp(prefix="phase2_smoke_")
    try:
        temp_ckpt = os.path.join(temp_dir, "test_ckpt.pt")
        save_phase2_checkpoint(test_model, opt, 1, 0.85, temp_ckpt)
        assert os.path.isfile(temp_ckpt), "Checkpoint file was not created!"

        # Verify restoration
        fresh_backbone = copy.deepcopy(model)
        fresh_head = StudentLinearNERHead(hidden_dim=N_EMBD, num_classes=len(BIO_LABELS))
        fresh_model = V1Phase2NERModel(fresh_backbone, fresh_head, freeze_backbone=False).to(DEVICE)
        restore_phase2_checkpoint(temp_ckpt, fresh_model)
        print("  --> Phase 2 checkpoint save/restore verified.")
    finally:
        shutil.rmtree(temp_dir, ignore_errors=True)

    # 5. Submission Packaging and Zip Structure
    print("[Check 5/6] Testing CodaBench ZIP packaging and validation...")
    temp_sub_dir = tempfile.mkdtemp(prefix="phase2_sub_smoke_")
    try:
        sample_doc_id = list(TEST_TEXTS.keys())[0] if TEST_TEXTS else "sample_test_doc_0"
        sample_preds = {
            sample_doc_id: [[0, 10, "COURT"]]
        }
        json_path, zip_path = package_codabench_submission(sample_preds, output_dir=temp_sub_dir)
        validate_submission_zip(zip_path, expected_doc_ids=[sample_doc_id])
        print("  --> Packaging and ZIP integrity verified.")
    finally:
        shutil.rmtree(temp_sub_dir, ignore_errors=True)

    # 6. Verify that the actual global Phase 1 weights did not change.
    print("[Check 6/6] Verifying Phase 1 model weights are preserved...")
    assert hasattr(model, "token_embedding"), "V1 backbone missing token_embedding"
    assert len(model.blocks) == N_LAYER, f"V1 backbone blocks count mismatch: {len(model.blocks)} != {N_LAYER}"
    for name, tensor in model.state_dict().items():
        if not torch.equal(tensor.detach().cpu(), original_backbone_state[name]):
            raise AssertionError(
                f"Preflight unexpectedly changed the original Phase 1 parameter: {name}"
            )
    print(f"  --> Original Phase 1 V1 weights unchanged ({N_LAYER} layers, N_EMBD={N_EMBD}).")

    print("=" * 70)
    print("ALL PRE-FLIGHT SMOKE TESTS PASSED CLEANLY! NOTEBOOK IS READY FOR EXP-1.")
    print("=" * 70)

run_exp1_preflight_smoke_tests()

## Controlled V1 5-fold out-of-fold ensemble

This run keeps the same Phase-1 V1 decoder-only backbone, tokenizer, Phase-2 token-BIO representation,
linear BIO head, full fine-tuning objective, learning rate (3e-5), batch size (8), and three-epoch maximum
used by the earlier full-fine-tuning branch. It changes the evaluation/training orchestration to five
document-disjoint folds and averages predictions from five independently adapted copies of the same
Phase-1 checkpoint.

Why: the earlier single holdout contained only about seven validation documents for twelve entity types.
That is a noisy basis for selecting decoding/calibration settings. Out-of-fold (OOF) predictions provide
held-out predictions for the entire labelled set without using a document to train the model that predicts it.

A baseline window decoder is compared with overlap-averaged token probabilities on all OOF documents.
Aggregation is selected only if it improves OOF strict span Macro-F1 by at least 0.005. We do not fit
per-label thresholds on the tiny validation split, and we do not refit a different model after selection.
The final cell creates exactly one ZIP.

This is a new ensemble experiment, not a claim that a score above 0.5 is guaranteed. Attach the original
Phase-1 V1 checkpoint and its matching tokenizer as inputs. Do not attach or load V2/V4 weights here.

In [ ]:
# EXPERIMENT: document-disjoint 5-fold cross-validation with a V1 ensemble.
# The model architecture and primary full-fine-tuning settings remain the V1 baseline settings.
import os, json, copy, random, gc, shutil, time, hashlib
from collections import Counter, defaultdict
import numpy as np
import torch
from torch.utils.data import DataLoader

CV_FOLDS = 5
CV_EPOCHS = 3
CV_LEARNING_RATE = 3e-5
CV_BATCH_SIZE = 8
CV_PIPELINE_VERSION = "v1_5fold_oof_ensemble_v3_fingerprinted_resume"
CV_OUTPUT_DIR = os.path.join(PHASE2_ROOT_DIR, "V1_5Fold_OOF_Ensemble")
CV_CONFIG_PATH = os.path.join(CV_OUTPUT_DIR, "cv_run_config.json")

def _sha256_file(path):
    if not path or not os.path.isfile(path):
        raise FileNotFoundError(f"Cannot fingerprint required file: {path!r}")
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def _sha256_json(value):
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()

# Prevent resumed folds from quietly mixing results from a different checkpoint, tokenizer, or data.
_CV_DATA_FINGERPRINT = _sha256_json({
    "texts": {str(k): v for k, v in TEXTS_BY_ID.items()},
    "annotations": {str(k): v for k, v in ANNOTATIONS_BY_ID.items()},
    "test_texts": {str(k): v for k, v in TEST_TEXTS.items()},
    "labels": list(LABELS),
})
CV_RUN_CONFIG = {
    "pipeline_version": CV_PIPELINE_VERSION,
    "folds": CV_FOLDS,
    "epochs_per_fold": CV_EPOCHS,
    "learning_rate": CV_LEARNING_RATE,
    "batch_size": CV_BATCH_SIZE,
    "context_length": MODEL_CONTEXT_LENGTH,
    "window_overlap": MODEL_CONTEXT_LENGTH // 2,
    "seed": SEED,
    "training_method": "V1 full fine-tuning, unweighted cross entropy",
    "phase1_checkpoint_sha256": _sha256_file(checkpoint_path),
    "phase1_tokenizer_sha256": _sha256_file(BPE_PATH),
    "phase2_data_sha256": _CV_DATA_FINGERPRINT,
    "label_order": list(LABELS),
}

# Resume completed folds when the exact same experiment configuration is rerun.
# If an existing output folder belongs to another configuration, fail rather than
# silently mixing checkpoints and predictions from different experiments.
if os.path.isdir(CV_OUTPUT_DIR) and os.path.isfile(CV_CONFIG_PATH):
    with open(CV_CONFIG_PATH, "r", encoding="utf-8") as _cf:
        _existing_cv_config = json.load(_cf)
    if _existing_cv_config != CV_RUN_CONFIG:
        raise RuntimeError(
            "An existing V1_5Fold_OOF_Ensemble folder uses a different configuration. "
            "Rename/archive that output folder before starting this configuration."
        )
elif os.path.isdir(CV_OUTPUT_DIR) and os.listdir(CV_OUTPUT_DIR):
    shutil.rmtree(CV_OUTPUT_DIR)
os.makedirs(CV_OUTPUT_DIR, exist_ok=True)
with open(CV_CONFIG_PATH, "w", encoding="utf-8") as _cf:
    json.dump(CV_RUN_CONFIG, _cf, indent=2)


def seed_phase2_run(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)


def make_iterative_document_folds(document_ids, annotations_by_id, n_splits=5, seed=42):
    """Deterministic multilabel-aware greedy document partitioning without extra dependencies."""
    ids = [str(x) for x in document_ids]
    if n_splits < 2 or len(ids) < n_splits:
        raise ValueError(f"Need at least {n_splits} documents for {n_splits} folds; found {len(ids)}.")
    rng = random.Random(seed)
    label_sets = {}
    total_presence = Counter()
    for doc_id in ids:
        labels_for_doc = {
            str(ent["label"] if isinstance(ent, dict) else ent[2])
            for ent in annotations_by_id.get(doc_id, [])
        }
        labels_for_doc &= set(LABELS)
        label_sets[doc_id] = labels_for_doc
        total_presence.update(labels_for_doc)

    folds = [[] for _ in range(n_splits)]
    fold_label_counts = [Counter() for _ in range(n_splits)]
    remaining = set(ids)
    max_size = int(np.ceil(len(ids) / n_splits))

    while remaining:
        remaining_presence = Counter()
        for doc_id in remaining:
            remaining_presence.update(label_sets[doc_id])
        active_labels = [lab for lab, cnt in remaining_presence.items() if cnt > 0]
        if active_labels:
            rare_label = min(active_labels, key=lambda lab: (remaining_presence[lab], lab))
            candidates = sorted(doc_id for doc_id in remaining if rare_label in label_sets[doc_id])
            rng.shuffle(candidates)
            candidates.sort(
                key=lambda doc_id: (
                    sum(1.0 / max(total_presence[lab], 1) for lab in label_sets[doc_id]),
                    len(label_sets[doc_id]),
                ),
                reverse=True,
            )
            chosen_doc = candidates[0]
            candidate_folds = [i for i, fold in enumerate(folds) if len(fold) < max_size]
            if not candidate_folds:
                candidate_folds = list(range(n_splits))
            chosen_fold = min(
                candidate_folds,
                key=lambda i: (fold_label_counts[i][rare_label], len(folds[i]), i),
            )
        else:
            # Annotation-free documents, if any, fill the smallest folds.
            chosen_doc = min(remaining, key=lambda doc_id: (len(label_sets[doc_id]), doc_id))
            candidate_folds = [i for i, fold in enumerate(folds) if len(fold) < max_size]
            if not candidate_folds:
                candidate_folds = list(range(n_splits))
            chosen_fold = min(candidate_folds, key=lambda i: (len(folds[i]), i))

        folds[chosen_fold].append(chosen_doc)
        fold_label_counts[chosen_fold].update(label_sets[chosen_doc])
        remaining.remove(chosen_doc)

    for fold in folds:
        fold.sort()
    flat = [doc_id for fold in folds for doc_id in fold]
    if len(flat) != len(set(flat)) or set(flat) != set(ids):
        raise AssertionError("Fold assignment lost or duplicated document IDs.")
    if max(map(len, folds)) - min(map(len, folds)) > 1:
        raise AssertionError(f"Fold sizes are unexpectedly imbalanced: {[len(x) for x in folds]}")
    return folds, fold_label_counts


ALL_LABELLED_DOC_IDS = [str(x) for x in TEXTS_BY_ID.keys()]
if len(ALL_LABELLED_DOC_IDS) < CV_FOLDS:
    raise RuntimeError("Too few labelled documents for the configured cross-validation folds.")
CV_DOCUMENT_FOLDS, CV_FOLD_LABEL_COUNTS = make_iterative_document_folds(
    ALL_LABELLED_DOC_IDS, ANNOTATIONS_BY_ID, n_splits=CV_FOLDS, seed=SEED
)
print("5-fold document-disjoint partition:")
for fi, fold in enumerate(CV_DOCUMENT_FOLDS, 1):
    fold_counts = Counter(ent["label"] for doc_id in fold for ent in ANNOTATIONS_BY_ID.get(doc_id, []))
    print(f"  Fold {fi}: {len(fold)} validation documents; entities=" + ", ".join(f"{lab}:{fold_counts[lab]}" for lab in LABELS if fold_counts[lab]))

# Fixed windowing/tokenizer settings are shared across every fold and final inference.
CV_PAD_ID = int(getattr(tokenizer, "pad_token_id", 0) or 0)
CV_WINDOW_OVERLAP = MODEL_CONTEXT_LENGTH // 2
CV_TEST_TEXTS = {str(k): v for k, v in TEST_TEXTS.items()}
CV_TEST_DATASET = Phase2NERDataset(
    document_ids=list(CV_TEST_TEXTS.keys()), text_by_id=CV_TEST_TEXTS, entities_by_id={},
    tokenizer_obj=tokenizer, max_length=MODEL_CONTEXT_LENGTH, window_overlap=CV_WINDOW_OVERLAP,
)
if len(CV_TEST_DATASET) == 0:
    raise RuntimeError("Fold-3 test set produced no windows; stopping before training.")


def collect_phase2_window_probabilities(model_obj, dataset, pad_token_id=0):
    """Collect per-window softmax probabilities plus original token offsets/indexes."""
    loader = DataLoader(
        dataset, batch_size=1, shuffle=False,
        collate_fn=lambda rows: collate_phase2_batch(rows, pad_token_id=pad_token_id),
    )
    grouped = defaultdict(list)
    model_obj.eval()
    with torch.inference_mode():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            logits = model_obj(input_ids, attention_mask=attention_mask)
            if logits.ndim != 3 or logits.shape[-1] != len(BIO_LABELS):
                raise RuntimeError(f"Unexpected BIO logits shape {tuple(logits.shape)}")
            probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()
            for i, raw_doc_id in enumerate(batch["document_ids"]):
                doc_id = str(raw_doc_id)
                offsets = list(batch["offsets"][i])
                seq_len = len(offsets)
                local_scores = np.asarray(probs[i, :seq_len], dtype=np.float32)
                if local_scores.shape != (seq_len, len(BIO_LABELS)):
                    raise AssertionError("Probability/offset length mismatch.")
                grouped[doc_id].append({
                    "token_start": int(batch["token_starts"][i]),
                    "offsets": offsets,
                    "scores": local_scores,
                })
    return dict(grouped)


def _span_key(span):
    return (int(span[0]), int(span[1]), str(span[2]))


def _dedupe_sort_span_list(spans):
    unique = {_span_key(span) for span in spans}
    return [list(x) for x in sorted(unique, key=lambda x: (x[0], x[1], x[2]))]


def aggregate_window_probabilities(records, doc_id):
    """Average the same global token's probabilities across all overlapping windows."""
    if not records:
        return np.zeros((0, len(BIO_LABELS)), dtype=np.float32), []
    records = sorted(records, key=lambda row: int(row["token_start"]))
    total_tokens = max(int(row["token_start"]) + len(row["offsets"]) for row in records)
    score_sum = np.zeros((total_tokens, len(BIO_LABELS)), dtype=np.float32)
    counts = np.zeros(total_tokens, dtype=np.int32)
    full_offsets = [None] * total_tokens
    for row in records:
        start = int(row["token_start"])
        scores = np.asarray(row["scores"], dtype=np.float32)
        offsets = row["offsets"]
        if scores.shape != (len(offsets), len(BIO_LABELS)):
            raise AssertionError(f"Bad token-score shape for document {doc_id}.")
        for local_idx, offset in enumerate(offsets):
            global_idx = start + local_idx
            if global_idx >= total_tokens:
                raise AssertionError("Window index exceeded the reconstructed token sequence.")
            score_sum[global_idx] += scores[local_idx]
            counts[global_idx] += 1
            if offset is None:
                continue
            if not isinstance(offset, (tuple, list)) or len(offset) < 2 or offset[0] is None or offset[1] is None:
                continue
            normalized = (int(offset[0]), int(offset[1]))
            if full_offsets[global_idx] is not None and full_offsets[global_idx] != normalized:
                raise RuntimeError(
                    f"Inconsistent offsets for document {doc_id}, token {global_idx}: "
                    f"{full_offsets[global_idx]} vs {normalized}"
                )
            full_offsets[global_idx] = normalized
    if np.any(counts == 0):
        raise RuntimeError(f"Windowing left token gaps in {doc_id}.")
    return score_sum / counts[:, None], full_offsets


def predict_phase2_from_window_records(records_by_doc, texts_by_id, mode="window"):
    """Decode either each window independently or one overlap-averaged document sequence."""
    predictions = {}
    for raw_doc_id, text in texts_by_id.items():
        doc_id = str(raw_doc_id)
        records = records_by_doc.get(doc_id, [])
        spans = []
        if mode == "window":
            for row in records:
                scores = np.asarray(row["scores"], dtype=np.float32)
                spans.extend(decode_token_bio_predictions(
                    torch.from_numpy(scores[None, :, :]), [row["offsets"]],
                    raw_text=text, id_to_bio=ID_TO_BIO, refine_boundaries=True,
                ))
        elif mode == "aggregate":
            scores, offsets = aggregate_window_probabilities(records, doc_id)
            if len(scores):
                spans.extend(decode_token_bio_predictions(
                    torch.from_numpy(scores[None, :, :]), [offsets],
                    raw_text=text, id_to_bio=ID_TO_BIO, refine_boundaries=True,
                ))
        else:
            raise ValueError(f"Unknown decoder mode: {mode}")
        predictions[doc_id] = _dedupe_sort_span_list(spans)
    return predictions


def validate_cv_prediction_mapping(predictions, texts_by_id, entity_labels):
    """Fail-closed validation of the exact document-ID/span schema before packaging."""
    expected_ids = {str(x) for x in texts_by_id}
    received_ids = {str(x) for x in predictions}
    if expected_ids != received_ids:
        raise ValueError(
            f"Prediction document IDs mismatch: missing={len(expected_ids-received_ids)}, "
            f"extra={len(received_ids-expected_ids)}."
        )
    allowed = set(entity_labels)
    for raw_doc_id, text in texts_by_id.items():
        doc_id = str(raw_doc_id)
        seen = set()
        for span in predictions.get(doc_id, []):
            if not isinstance(span, (list, tuple)) or len(span) < 3:
                raise TypeError(f"Invalid span representation for {doc_id}: {span!r}")
            start, end, label = span[:3]
            if isinstance(start, bool) or isinstance(end, bool) or not isinstance(start, (int, np.integer)) or not isinstance(end, (int, np.integer)):
                raise TypeError(f"Span offsets must be integers for {doc_id}: {span!r}")
            if not (0 <= int(start) < int(end) <= len(text)):
                raise ValueError(f"Out-of-range span for {doc_id} (text length {len(text)}): {span!r}")
            if str(label) not in allowed:
                raise ValueError(f"Unknown entity label for {doc_id}: {label!r}")
            key = (int(start), int(end), str(label))
            if key in seen:
                raise ValueError(f"Duplicate predicted entity for {doc_id}: {key!r}")
            seen.add(key)
    return True


def score_phase2_predictions(predictions, texts_by_id, gold_by_id):
    doc_ids = [str(x) for x in texts_by_id]
    normalized_gold = {doc_id: gold_by_id.get(doc_id, []) for doc_id in doc_ids}
    normalized_predictions = {doc_id: predictions.get(doc_id, []) for doc_id in doc_ids}
    metrics = strict_span_metrics(
        gold_by_id=normalized_gold, predicted_by_id=normalized_predictions,
        labels=LABELS, document_ids=doc_ids,
    )
    per_label = metrics["per_label"]
    metrics["macro_precision"] = float(np.mean([per_label[lab]["precision"] for lab in LABELS]))
    metrics["macro_recall"] = float(np.mean([per_label[lab]["recall"] for lab in LABELS]))
    metrics["tp"] = int(sum(per_label[lab]["tp"] for lab in LABELS))
    metrics["fp"] = int(sum(per_label[lab]["fp"] for lab in LABELS))
    metrics["fn"] = int(sum(per_label[lab]["fn"] for lab in LABELS))
    return metrics


def summarize_metrics(metrics):
    return {key: value for key, value in metrics.items() if key not in ("per_label", "predictions_by_doc")}


print("Cross-validation, probability collection, overlap aggregation, and exact-span metrics are ready.")


In [ ]:
# CPU-friendly preflight tests for fold partitioning and overlap aggregation.
# These tests do not access test labels, modify weights, or start training.
import numpy as np

_test_doc_ids = [f"d{i}" for i in range(15)]
_test_ann = {
    "d0": [{"start": 0, "end": 1, "label": "APP"}, {"start": 3, "end": 4, "label": "DATE"}],
    "d1": [{"start": 0, "end": 1, "label": "RESP"}],
    "d2": [{"start": 0, "end": 1, "label": "APP"}],
    "d3": [{"start": 0, "end": 1, "label": "COURT"}],
    "d4": [{"start": 0, "end": 1, "label": "DATE"}],
    "d5": [{"start": 0, "end": 1, "label": "COURT"}],
    **{f"d{i}": [] for i in range(6, 15)},
}
_test_folds, _ = make_iterative_document_folds(_test_doc_ids, _test_ann, n_splits=5, seed=11)
assert set(x for fold in _test_folds for x in fold) == set(_test_doc_ids)
assert sum(len(fold) for fold in _test_folds) == len(_test_doc_ids)
assert max(map(len, _test_folds)) - min(map(len, _test_folds)) <= 1

# Two overlapping windows: global token 1 should average probability vectors across windows.
_prob_a = np.array([[0.8] + [0.2/ (len(BIO_LABELS)-1)]*(len(BIO_LABELS)-1),
                    [0.7] + [0.3/(len(BIO_LABELS)-1)]*(len(BIO_LABELS)-1)], dtype=np.float32)
_prob_b = np.array([[0.1] + [0.9/(len(BIO_LABELS)-1)]*(len(BIO_LABELS)-1),
                    [0.6] + [0.4/(len(BIO_LABELS)-1)]*(len(BIO_LABELS)-1)], dtype=np.float32)
_synthetic_records = [
    {"token_start": 0, "offsets": [(0, 1), (1, 2)], "scores": _prob_a},
    {"token_start": 1, "offsets": [(1, 2), (2, 3)], "scores": _prob_b},
]
_agg_scores, _agg_offsets = aggregate_window_probabilities(_synthetic_records, "synthetic")
assert _agg_scores.shape == (3, len(BIO_LABELS))
assert _agg_offsets == [(0, 1), (1, 2), (2, 3)]
assert np.allclose(_agg_scores[1], (_prob_a[1] + _prob_b[0]) / 2.0, atol=1e-6)


# Submission span validation: exact one-character offsets are allowed; bad bounds and duplicates fail closed.
assert validate_cv_prediction_mapping({"toy": [[0, 1, "APP"]]}, {"toy": "A"}, LABELS)
try:
    validate_cv_prediction_mapping({"toy": [[0, 2, "APP"]]}, {"toy": "A"}, LABELS)
    raise AssertionError("Out-of-bounds prediction was not rejected.")
except ValueError:
    pass
try:
    validate_cv_prediction_mapping({"toy": [[0, 1, "APP"], [0, 1, "APP"]]}, {"toy": "A"}, LABELS)
    raise AssertionError("Duplicate prediction was not rejected.")
except ValueError:
    pass

print("Preflight PASS: fold integrity, near-equal fold sizes, overlap averaging, offset consistency, and span-schema validation.")


In [ ]:
# Train five independent V1 full-fine-tuning models; collect held-out OOF and test probabilities.
# Completed folds are cached on disk so a disconnected/interrupted Kaggle run can resume without repeating them.
import pickle
cv_histories = []
oof_window_records_by_doc = {}
test_window_probability_accumulator = {}

for fold_idx, val_ids in enumerate(CV_DOCUMENT_FOLDS, start=1):
    val_id_set = set(val_ids)
    train_ids = [doc_id for doc_id in ALL_LABELLED_DOC_IDS if doc_id not in val_id_set]
    if set(train_ids) & val_id_set:
        raise AssertionError(f"Train/validation overlap in fold {fold_idx}.")
    fold_dir = os.path.join(CV_OUTPUT_DIR, f"fold_{fold_idx:02d}")
    fold_checkpoint_dir = os.path.join(fold_dir, "checkpoints")
    fold_checkpoint_path = os.path.join(fold_checkpoint_dir, "best_v1_full_ft.pt")
    fold_cache_path = os.path.join(fold_dir, "fold_prediction_cache.pkl")
    fold_history_path = os.path.join(fold_dir, "fold_history.json")

    cache_is_valid = False
    if os.path.isfile(fold_checkpoint_path) and os.path.isfile(fold_cache_path) and os.path.isfile(fold_history_path):
        try:
            with open(fold_cache_path, "rb") as _cf:
                _fold_cache = pickle.load(_cf)
            cache_is_valid = (
                _fold_cache.get("fold") == fold_idx
                and _fold_cache.get("validation_document_ids") == list(val_ids)
                and set(_fold_cache.get("oof_records", {}).keys()) == set(val_ids)
                and _fold_cache.get("config") == CV_RUN_CONFIG
                and _fold_cache.get("fold_checkpoint_sha256") == _sha256_file(fold_checkpoint_path)
            )
        except Exception as _cache_exc:
            print(f"Fold {fold_idx}: prediction cache not usable ({_cache_exc}); retraining this fold.")
            cache_is_valid = False

    if cache_is_valid:
        fold_history = _fold_cache["history"]
        fold_best_f1 = float(_fold_cache["best_validation_strict_macro_f1"])
        fold_checkpoint_state = torch.load(fold_checkpoint_path, map_location="cpu", weights_only=False)
        fold_oof_records = _fold_cache["oof_records"]
        fold_test_records = _fold_cache["test_records"]
        print(f"\nFOLD {fold_idx}/{CV_FOLDS}: reusing verified checkpoint + prediction cache; no retraining.")
    else:
        print("\n" + "=" * 78)
        print(f"FOLD {fold_idx}/{CV_FOLDS}: {len(train_ids)} train documents, {len(val_ids)} held-out documents")
        print("=" * 78)
        # An incomplete fold folder is removed; other completed folds are preserved.
        if os.path.isdir(fold_dir):
            shutil.rmtree(fold_dir)
        os.makedirs(fold_checkpoint_dir, exist_ok=True)

        fold_train_dataset = Phase2NERDataset(
            document_ids=train_ids, text_by_id=TEXTS_BY_ID, entities_by_id=ANNOTATIONS_BY_ID,
            tokenizer_obj=tokenizer, max_length=MODEL_CONTEXT_LENGTH, window_overlap=CV_WINDOW_OVERLAP,
        )
        fold_val_dataset = Phase2NERDataset(
            document_ids=val_ids, text_by_id=TEXTS_BY_ID, entities_by_id=ANNOTATIONS_BY_ID,
            tokenizer_obj=tokenizer, max_length=MODEL_CONTEXT_LENGTH, window_overlap=CV_WINDOW_OVERLAP,
        )
        if not len(fold_train_dataset) or not len(fold_val_dataset):
            raise RuntimeError(f"Empty train/validation windows in fold {fold_idx}.")
        print(f"Windows: {len(fold_train_dataset)} train / {len(fold_val_dataset)} validation")

        seed_phase2_run(SEED + 1009 * fold_idx)
        fold_model = V1Phase2NERModel(
            backbone=copy.deepcopy(model),
            ner_head=StudentLinearNERHead(hidden_dim=N_EMBD, num_classes=len(BIO_LABELS)),
            freeze_backbone=False,
        ).to(DEVICE)

        fold_history, fold_best_f1 = run_phase2_training(
            fold_model, fold_train_dataset, fold_val_dataset,
            TEXTS_BY_ID, ANNOTATIONS_BY_ID,
            epochs=CV_EPOCHS, lr=CV_LEARNING_RATE, batch_size=CV_BATCH_SIZE,
            checkpoint_path=fold_checkpoint_path, device=DEVICE, class_weights=None,
        )
        if not os.path.isfile(fold_checkpoint_path):
            raise RuntimeError(f"Fold {fold_idx} did not produce a best checkpoint.")
        fold_checkpoint_state = restore_phase2_checkpoint(fold_checkpoint_path, fold_model)

        # Each document is collected only from its held-out fold model.
        fold_oof_records = collect_phase2_window_probabilities(
            fold_model, fold_val_dataset, pad_token_id=CV_PAD_ID
        )
        for doc_id in val_ids:
            fold_oof_records.setdefault(str(doc_id), [])

        fold_test_records = collect_phase2_window_probabilities(
            fold_model, CV_TEST_DATASET, pad_token_id=CV_PAD_ID
        )
        fold_cache = {
            "fold": fold_idx,
            "validation_document_ids": list(val_ids),
            "config": CV_RUN_CONFIG,
            "history": fold_history,
            "best_validation_strict_macro_f1": float(fold_best_f1),
            "fold_checkpoint_sha256": _sha256_file(fold_checkpoint_path),
            "oof_records": fold_oof_records,
            "test_records": fold_test_records,
        }
        with open(fold_cache_path, "wb") as _cf:
            pickle.dump(fold_cache, _cf, protocol=pickle.HIGHEST_PROTOCOL)
        with open(fold_history_path, "w", encoding="utf-8") as _cf:
            json.dump({
                "fold": fold_idx,
                "validation_document_ids": list(val_ids),
                "history": fold_history,
                "best_validation_strict_macro_f1": float(fold_best_f1),
                "best_checkpoint_epoch": int(fold_checkpoint_state.get("epoch", -1)),
                "checkpoint_path": fold_checkpoint_path,
            }, _cf, indent=2, ensure_ascii=False)
        del fold_model, fold_train_dataset, fold_val_dataset
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Ensure one out-of-fold prediction collection per document.
    for doc_id in val_ids:
        doc_id = str(doc_id)
        if doc_id in oof_window_records_by_doc:
            raise AssertionError(f"Document {doc_id} received OOF predictions more than once.")
        oof_window_records_by_doc[doc_id] = fold_oof_records.get(doc_id, [])

    # Add this fold's window probabilities into a running ensemble accumulator.
    for doc_id, records in fold_test_records.items():
        for row in records:
            token_start = int(row["token_start"])
            seq_len = len(row["offsets"])
            key = (str(doc_id), token_start, seq_len)
            if key not in test_window_probability_accumulator:
                test_window_probability_accumulator[key] = {
                    "offsets": list(row["offsets"]),
                    "score_sum": np.asarray(row["scores"], dtype=np.float32).copy(),
                    "model_count": 1,
                }
            else:
                item = test_window_probability_accumulator[key]
                if item["offsets"] != list(row["offsets"]):
                    raise RuntimeError(f"Fold-specific offsets differ for test window {key}.")
                if item["score_sum"].shape != np.asarray(row["scores"]).shape:
                    raise RuntimeError(f"Fold-specific score dimensions differ for test window {key}.")
                item["score_sum"] += np.asarray(row["scores"], dtype=np.float32)
                item["model_count"] += 1

    cv_histories.append({
        "fold": fold_idx,
        "train_documents": len(train_ids),
        "validation_documents": len(val_ids),
        "history": fold_history,
        "best_validation_strict_macro_f1": float(fold_best_f1),
        "best_checkpoint_epoch": int(fold_checkpoint_state.get("epoch", -1)),
        "checkpoint_path": fold_checkpoint_path,
        "resumed_from_cache": bool(cache_is_valid),
    })
    print(f"Fold {fold_idx} complete. Best single-fold validation F1={fold_best_f1:.4f}")
    del fold_oof_records, fold_test_records
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if set(oof_window_records_by_doc) != set(ALL_LABELLED_DOC_IDS):
    missing = sorted(set(ALL_LABELLED_DOC_IDS) - set(oof_window_records_by_doc))
    extra = sorted(set(oof_window_records_by_doc) - set(ALL_LABELLED_DOC_IDS))
    raise AssertionError(f"OOF coverage mismatch: {len(missing)} missing, {len(extra)} extra.")

_bad_test_windows = [(key, item["model_count"]) for key, item in test_window_probability_accumulator.items()
                     if item["model_count"] != CV_FOLDS]
if _bad_test_windows:
    raise RuntimeError(f"Some test windows were not scored by all {CV_FOLDS} models: {_bad_test_windows[:5]}")

ensemble_test_window_records_by_doc = defaultdict(list)
for (doc_id, token_start, seq_len), item in test_window_probability_accumulator.items():
    ensemble_test_window_records_by_doc[doc_id].append({
        "token_start": token_start,
        "offsets": item["offsets"],
        "scores": item["score_sum"] / float(item["model_count"]),
    })
for doc_id in CV_TEST_TEXTS:
    ensemble_test_window_records_by_doc.setdefault(str(doc_id), [])
for doc_id in ensemble_test_window_records_by_doc:
    ensemble_test_window_records_by_doc[doc_id].sort(key=lambda row: row["token_start"])

with open(os.path.join(CV_OUTPUT_DIR, "fold_training_history.json"), "w", encoding="utf-8") as f:
    json.dump(cv_histories, f, indent=2, ensure_ascii=False)
print("\nAll folds trained or resumed. OOF documents:", len(oof_window_records_by_doc))
print("Test windows averaged across all fold models:", len(test_window_probability_accumulator))


### Final model-selection rule

The notebook uses the fold-wise held-out predictions to compare the original per-window BIO decoder with overlap-averaged token probabilities. It selects aggregation only when the difference in out-of-fold strict span Macro-F1 is at least 0.005. This reduces the chance of selecting a more complex decoder based on tiny validation noise. Per-label calibration is deliberately omitted for this run: with only 70 labelled documents, a 12-label coordinate search can overfit even though cross-validation improves the estimate.

In [ ]:
# Select decoding on out-of-fold predictions, ensemble all five models for test, and package ONE ZIP.
import os, json, zipfile

OOF_TEXTS = {str(k): v for k, v in TEXTS_BY_ID.items()}
OOF_GOLD = {str(k): v for k, v in ANNOTATIONS_BY_ID.items()}

# Compare decoders on out-of-fold documents only. Each OOF document was predicted by a model
# that was not trained on that document. This is more informative than a seven-document holdout.
oof_window_predictions = predict_phase2_from_window_records(
    oof_window_records_by_doc, OOF_TEXTS, mode="window"
)
oof_aggregate_predictions = predict_phase2_from_window_records(
    oof_window_records_by_doc, OOF_TEXTS, mode="aggregate"
)
oof_window_metrics = score_phase2_predictions(oof_window_predictions, OOF_TEXTS, OOF_GOLD)
oof_aggregate_metrics = score_phase2_predictions(oof_aggregate_predictions, OOF_TEXTS, OOF_GOLD)

print("\nOUT-OF-FOLD DECODING COMPARISON (all labelled documents; each prediction is held out)")
print(f"Window-wise decode:       Strict Macro-F1={oof_window_metrics['strict_macro_f1']:.4f} "
      f"P={oof_window_metrics['macro_precision']:.4f} R={oof_window_metrics['macro_recall']:.4f} "
      f"TP={oof_window_metrics['tp']} FP={oof_window_metrics['fp']} FN={oof_window_metrics['fn']}")
print(f"Overlap-averaged decode:  Strict Macro-F1={oof_aggregate_metrics['strict_macro_f1']:.4f} "
      f"P={oof_aggregate_metrics['macro_precision']:.4f} R={oof_aggregate_metrics['macro_recall']:.4f} "
      f"TP={oof_aggregate_metrics['tp']} FP={oof_aggregate_metrics['fp']} FN={oof_aggregate_metrics['fn']}")

if float(oof_aggregate_metrics["strict_macro_f1"]) >= float(oof_window_metrics["strict_macro_f1"]) + 0.005:
    SELECTED_OOF_DECODER = "aggregate"
    SELECTED_OOF_METRICS = oof_aggregate_metrics
    print("Selected overlap aggregation: it clears the 0.005 OOF Strict Macro-F1 improvement margin.")
else:
    SELECTED_OOF_DECODER = "window"
    SELECTED_OOF_METRICS = oof_window_metrics
    print("Kept baseline window decoding: aggregation did not clear the 0.005 OOF improvement margin.")

# Test prediction uses per-window probabilities averaged across all five fold models, then the
# same decoder choice made from OOF data. There is no full-data refit and no validation-fitted
# per-label calibration transferred to a different model.
TEST_PREDICTIONS = predict_phase2_from_window_records(
    ensemble_test_window_records_by_doc, CV_TEST_TEXTS, mode=SELECTED_OOF_DECODER
)
for doc_id in CV_TEST_TEXTS:
    TEST_PREDICTIONS.setdefault(str(doc_id), [])

validate_cv_prediction_mapping(
    predictions=TEST_PREDICTIONS,
    texts_by_id=CV_TEST_TEXTS,
    entity_labels=LABELS,
)

# Persist OOF metrics and span errors for the next decision, not a second submission.
selected_oof_predictions = oof_aggregate_predictions if SELECTED_OOF_DECODER == "aggregate" else oof_window_predictions
_errors = []
for doc_id, text in OOF_TEXTS.items():
    gold_spans = {
        (int(ent["start"]), int(ent["end"]), str(ent["label"])) if isinstance(ent, dict)
        else (int(ent[0]), int(ent[1]), str(ent[2]))
        for ent in OOF_GOLD.get(doc_id, [])
    }
    pred_spans = {_span_key(span) for span in selected_oof_predictions.get(doc_id, [])}
    for span in sorted(gold_spans - pred_spans):
        s, e, lab = span
        _errors.append({"document_id": doc_id, "error_type": "false_negative", "span": [s, e, lab],
                        "entity_text": text[s:e], "context": text[max(0, s-100):min(len(text), e+100)]})
    for span in sorted(pred_spans - gold_spans):
        s, e, lab = span
        _errors.append({"document_id": doc_id, "error_type": "false_positive", "span": [s, e, lab],
                        "entity_text": text[s:e], "context": text[max(0, s-100):min(len(text), e+100)]})

with open(os.path.join(CV_OUTPUT_DIR, "oof_validation_metrics.json"), "w", encoding="utf-8") as f:
    json.dump({
        "folds": CV_FOLDS,
        "oof_documents": len(OOF_TEXTS),
        "window_decoder": summarize_metrics(oof_window_metrics) | {"per_label": oof_window_metrics["per_label"]},
        "aggregate_decoder": summarize_metrics(oof_aggregate_metrics) | {"per_label": oof_aggregate_metrics["per_label"]},
        "selected_decoder": SELECTED_OOF_DECODER,
        "selected_oof_metrics": summarize_metrics(SELECTED_OOF_METRICS),
        "notes": "OOF predictions only; each document is held out from the fold model that predicts it. No test labels used.",
    }, f, indent=2, ensure_ascii=False)
with open(os.path.join(CV_OUTPUT_DIR, "oof_validation_span_errors.json"), "w", encoding="utf-8") as f:
    json.dump({"selected_decoder": SELECTED_OOF_DECODER, "total_errors": len(_errors), "errors": _errors[:5000]},
              f, indent=2, ensure_ascii=False)

# Exactly one ZIP for this experiment.
prediction_path, submission_zip_path = package_codabench_submission(
    predictions=TEST_PREDICTIONS,
    output_dir=CV_OUTPUT_DIR,
    zip_filename="submission_v1_5fold_oof_ensemble.zip",
)
validate_submission_zip(submission_zip_path, expected_doc_ids=CV_TEST_TEXTS.keys())

pred_counts = Counter(span[2] for spans in TEST_PREDICTIONS.values() for span in spans)
manifest = {
    "experiment": "V1_5Fold_OOF_Ensemble",
    "backbone": "Original Phase-1 V1 decoder-only Transformer",
    "training": {"folds": CV_FOLDS, "epochs_per_fold": CV_EPOCHS, "learning_rate": CV_LEARNING_RATE,
                 "batch_size": CV_BATCH_SIZE, "loss": "unweighted CrossEntropyLoss with IGNORE_INDEX",
                 "no_full_data_refit": True},
    "selected_decoder": SELECTED_OOF_DECODER,
    "oof_window_f1": float(oof_window_metrics["strict_macro_f1"]),
    "oof_aggregate_f1": float(oof_aggregate_metrics["strict_macro_f1"]),
    "selected_oof_f1": float(SELECTED_OOF_METRICS["strict_macro_f1"]),
    "test_documents": len(CV_TEST_TEXTS),
    "test_predicted_entities": int(sum(len(v) for v in TEST_PREDICTIONS.values())),
    "test_entities_by_label": {label: int(pred_counts.get(label, 0)) for label in LABELS},
    "submission_zip": os.path.basename(submission_zip_path),
    "zip_members": ["prediction.json"],
    "note": "The 0.3223 score is the prior leaderboard reference. This run has not been evaluated on Codabench yet.",
}
with open(os.path.join(CV_OUTPUT_DIR, "experiment_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

zip_outputs = [name for name in os.listdir(CV_OUTPUT_DIR) if name.lower().endswith(".zip")]
assert zip_outputs == ["submission_v1_5fold_oof_ensemble.zip"], f"Expected exactly one ZIP, found: {zip_outputs}"
print("\n" + "=" * 72)
print("V1 FIVE-FOLD OOF ENSEMBLE COMPLETE")
print("Selected decoder:", SELECTED_OOF_DECODER)
print(f"OOF Strict Macro-F1: {SELECTED_OOF_METRICS['strict_macro_f1']:.4f}")
print("OOF diagnostic metrics:", os.path.join(CV_OUTPUT_DIR, "oof_validation_metrics.json"))
print("OOF exact-span errors:", os.path.join(CV_OUTPUT_DIR, "oof_validation_span_errors.json"))
print("Submission ZIP:", submission_zip_path)
print("Only one submission ZIP was created.")


## Run sequence

1. Attach the original `v1_model.pt`, matching `bpe_en_25k.json`, and the Phase 2 fold files.
2. Run the notebook from the beginning so the pretrained Phase-1 V1 checkpoint and tokenizer are restored cleanly.
3. Run the preflight tests before starting the fold training.
4. Execute the five-fold training cell. This takes longer than the former single run because five independent adapters/heads are trained from the same Phase-1 checkpoint.
5. Inspect `oof_validation_metrics.json` and `oof_validation_span_errors.json` before submitting.
6. Download only `submission_v1_5fold_oof_ensemble.zip` from `V1_5Fold_OOF_Ensemble`.
7. If interrupted, rerunning the same cells resumes only completed folds whose checkpoint, tokenizer, data, and training configuration hashes match exactly. If any differ, the notebook refuses to mix artifacts; archive the old output directory before a fresh run.

The run writes one ZIP. It does not perform an all-data refit, does not reuse calibration thresholds from another checkpoint, and does not submit automatically to Codabench.